In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os, sys, glob, warnings
from scipy.stats import ttest_1samp, wilcoxon, binomtest
from scipy import stats
try:
    from matplotlib.collections import BrokenBarHCollection
except ImportError:
    BrokenBarHCollection = None
import pandas as pd
import numpy as np
import os, glob
from scipy.stats import ttest_1samp, ttest_ind, t as t_dist

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

import re

from collections import defaultdict

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
# === COLOR DEFINITIONS ===
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33); c2 = (1.00, 0.23, 0.19)
c3 = (1.00, 0.58, 0.00); c4 = (1.00, 0.80, 0.00)
c5 = (0.30, 0.85, 0.39); c6 = (0.35, 0.78, 0.98)
c7 = (0.20, 0.67, 0.86); c8 = (0.00, 0.48, 1.00)
c9 = (0.35, 0.34, 0.84); c10 = (0.00, 0.31, 0.57)

orange1='#feedde'; orange2='#fdbe85'; orange3='#fd8d3c'; orange4='#e6550d'; orange5='#a63603'
blue1='#eff3ff'; blue2='#bdd7e7'; blue3='#6baed6'; blue4='#3182bd'; blue5='#08519c'
green1='#edf8e9'; green2='#bae4b3'; green3='#74c476'; green4='#31a354'; green5='#006d2c'
grey1='#f7f7f7'; grey2='#cccccc'; grey3='#969696'; grey4='#636363'; grey5='#252525'
purple1='#f2f0f7'; purple2='#cbc9e2'; purple3='#9e9ac8'; purple4='#756bb1'; purple5='#54278f'
red1='#fee5d9'; red2='#fcae91'; red3='#fb6a4a'; red4='#de2d26'; red5='#a50f15'

# 1. Configuration

In [ ]:
ideogram_file = 'chromosome_ideogram_hg19.txt'

chromosome_sizes = {
    'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276,
    'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022,
    'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895,
    'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753,
    'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520,
    'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560
}

all_chromosomes = ['chr1','chr2','chr3','chr4','chr5','chr6','chr7','chr8','chr9',
                   'chr10','chr11','chr12','chr13','chr14','chr15','chr16','chr17',
                   'chr18','chr19','chr20','chr21','chr22','chrX']

In [ ]:
def ideograms(ideogram_file, chromosome):
    
    color_lookup = {'gneg': (1., 1., 1.),
                    'gpos25': (.6, .6, .6),
                    'gpos50': (.4, .4, .4),
                    'gpos75': (.2, .2, .2),
                   'gpos100': (0., 0., 0.),
                      'acen': (.8, .4, .4),
                      'gvar': (.8, .8, .8),
                     'stalk': (.9, .9, .9)}
    
    ideogram = open(ideogram_file)
    ideogram.readline()
    xranges = []
    colors = []
    mid_points = []
    labels = []

    for line in ideogram:
        chrom, start, stop, label, stain = line.strip().split('\t')
        start = int(start)
        stop = int(stop)
        width = stop - start
        mid_point = start + (width/2)
        if chrom == chromosome:
            xranges.append((start, width))
            colors.append(color_lookup[stain])
            mid_points.append(mid_point)
            labels.append(label)
        
    return xranges, [0, 0.9], colors, mid_points, labels

def plot_chromosome(ideogram_file, chromosome, ax):

    xranges, yrange, colors, midpoints, labels = ideograms(ideogram_file, chromosome)

    ax.broken_barh(xranges, yrange, facecolors= colors, edgecolor = 'black')

    ax.set_xticks(midpoints)
    ax.set_xticklabels(labels, rotation = 90, fontsize = 9)
    ax.set_yticks([])
    ax.text(-0.013, 0.35, chromosome, transform=ax.transAxes, fontsize = 15, ha = 'right')
    ax.xaxis.set_tick_params(width=0.8, color = grey3, length = 6)

    ax.minorticks_off()

    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    return ax

def load_centromeres(filepath):
    """
    Parses the UCSC cytoband file to find the start/end of centromeres ('acen').
    Returns a dictionary: {'1': (start, end), '2': (start, end), ...}
    """
    
    df = pd.read_csv(filepath, sep="\t", comment='#', header=None, 
                     names=['chrom', 'start', 'end', 'name', 'type'])
    
    # Filter for centromeric regions ('acen')
    acen = df[df['type'] == 'acen'].copy()
    
    centromeres = {}
    for chrom, grp in acen.groupby('chrom'):
        # Centromeres usually span two bands (p-arm end, q-arm start)
        # We take the overall min start and max end.
        start = grp['start'].min()
        end = grp['end'].max()
        
        centromeres[chrom] = {'start': start, 'end': end}
        
    return centromeres

centromere_dict = load_centromeres('chromosome_ideogram_hg19.txt')
print("Loaded centromeres for:", list(centromere_dict.keys()))

# Simulated samples

## 1.  Simulated samples:  Using ground-truth phasing & coordinates to call mCA cell fraction

- Uses ground-truth mCA coordinates and SNP phasing (i.e. that stored in the simulated data files) to calculate cell fraction of mCA.
- i.e. with perfect phasing and perfect coordinates, what is detectable at each cell fraction?

For each simulated sample independently: 
1) Load that sample's SNP file
2) Filter to phased hets in the ground-truth region (Start_bp to End_bp)
3) Compute phased_dev = (2 × Phased_Haplotype - 1) × (VAF - 0.5) using that sample's own column
4) T-test + permutation test → significant or not

This is the "perfect information ceiling": what could you detect if you had perfect phasing and knew the exact mCA boundaries? No families,no reference cascade, no detections file needed.

Requires:
- simulation_manifest.csv
- Simulated SNP files (with Phased_Haplotype column)

### Configuration

In [ ]:
TRUTH_CSV = 'CNV_panel_simulated_Samples_Feb2026/mCA_caller_simulation_results_TEST_set_v13_caller_all_cell_fractions/truth_comparison.csv'
MANIFEST_CSV = 'CNV_panel_simulated_Samples_Feb2026/Simulated_samples_TEST_set/simulation_manifest.csv'
SIMULATED_DIR = 'CNV_panel_simulated_Samples_Feb2026/Simulated_samples_TEST_set'
OUTPUT_CSV = 'CNV_panel_simulated_Samples_Feb2026/mCA_caller_simulation_results_TEST_set_v13_caller_all_cell_fractions/phased_per_sample_results.csv'
OUTPUT_DIR = 'CNV_panel_simulated_Samples_Feb2026/mCA_caller_simulation_results_TEST_set_v13_caller_all_cell_fractions'

# Het filter for target samples
TARGET_HET_LO = 0.20
TARGET_HET_HI = 0.80
HET_LO = 0.02
HET_HI = 0.98

# Significance threshold
P_THRESHOLD = 0.05

# Minimum phased het SNPs required
MIN_HETS = 3

# PON configuration
PON_BASE_PATH = "."  # adjust if notebook isn't in the same folder
PON_LIBRARIES = ['SLX_19285', 'SLX_20125', 'SLX_20127']

# Read PON sample list from master matrix
_pon_matrix = pd.read_csv(
    'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv',
    sep='\t', nrows=0
)
FINAL_TIMEPOINT_CONTROLS = [c for c in _pon_matrix.columns if c.startswith('CNTRL')]

# Known germline mCA chromosomes — skip these
KNOWN_GERMLINE_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

DEBUG = True

# Permutation settings
N_PERMUTATIONS = 10000
PERM_SEED = 42

### Cell fraction estimation

In [ ]:
def phased_dev_to_cf(pd_val, event):
    """Convert mean phased_dev to cell fraction estimate."""
    if pd_val <= 0:
        return 0.0
    if event in ('CN-LOH', 'CNLOH'):
        return float(np.clip(2 * pd_val, 0, 1))
    elif event == 'GAIN':
        return float(np.clip(4 * pd_val / (1 - 2 * pd_val) if pd_val < 0.5 else 1.0, 0, 1))
    elif event == 'LOSS':
        return float(np.clip(4 * pd_val / (1 + 2 * pd_val), 0, 1))
    else:
        return float(np.clip(2 * pd_val, 0, 1))

def analyse_single_sample(snp_data, chrom, region_start, region_end, event,
                           het_lo=0.02, het_hi=0.98, min_hets=3,
                           p_threshold=0.05, n_permutations=10000, perm_seed=42):
    
    # print("USING UPDATED VERSION")  # ← add this temporarily
    """
    Analyse one sample using its own Phased_Haplotype and ground-truth coordinates.
    
    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF, Phased_Haplotype columns
    chrom, region_start, region_end : region coordinates
    event : str ('CN-LOH', 'GAIN', 'LOSS')
    het_lo, het_hi : VAF range for het filter
    min_hets : minimum phased het SNPs required
    p_threshold : significance threshold for one-sided t-test
    n_permutations : number of sign-flip permutations
    perm_seed : random seed for permutation test
    
    Returns dict with results or None if insufficient data.
    """
    # Filter to chromosome
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == str(chrom)].copy()
    
    # Het filter
    hets = chrom_snps[
        (chrom_snps['VAF'] >= het_lo) & (chrom_snps['VAF'] <= het_hi)
    ].copy()
    
    if len(hets) == 0:
        return None
    
    # Must have Phased_Haplotype column
    if 'Phased_Haplotype' not in hets.columns:
        return None
    
    # Get phased hets in region
    ph = pd.to_numeric(hets['Phased_Haplotype'], errors='coerce')
    region_mask = (hets['position'] >= region_start) & (hets['position'] <= region_end)
    phased_region = hets[region_mask & ph.notna()].copy()
    
    n_hets = len(phased_region)
    if n_hets < min_hets:
        return None
    
    # Compute phased_dev using each SNP's own Phased_Haplotype
    # Phased_Haplotype=1 → ALT on amplified haplotype → expect VAF > 0.5
    # Phased_Haplotype=0 → ALT on other haplotype → expect VAF < 0.5
    # phased_dev = (2*phase - 1) * (VAF - 0.5)
    #   → positive when VAF shifts in expected direction
    #   → negative when noise pushes VAF the wrong way
    phase_vals = pd.to_numeric(phased_region['Phased_Haplotype'], errors='coerce').values
    vaf_vals = phased_region['VAF'].astype(float).values
    phased_dev = (2 * phase_vals - 1) * (vaf_vals - 0.5)
    if event == 'LOSS':
        phased_dev = -phased_dev
        # print(f"LOSS flip applied — mean_phased_dev = {np.mean(phased_dev):.6f}")

    mean_dev = float(np.mean(phased_dev))
    std_dev = float(np.std(phased_dev, ddof=1))
    se_dev = std_dev / np.sqrt(n_hets)
    
    # One-sample t-test: is mean phased_dev > 0?
    t_stat, p_two = ttest_1samp(phased_dev, 0)
    p_onesided = float(p_two / 2) if t_stat > 0 else float(1 - p_two / 2)
    
    # Permutation test: randomly flip signs
    rng = np.random.default_rng(perm_seed)
    abs_vals = np.abs(phased_dev)
    null_means = np.array([
        np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
        for _ in range(n_permutations)
    ])
    p_permutation = float(np.mean(null_means >= mean_dev))
    
    # CF estimate + 95% CI
    cf = phased_dev_to_cf(mean_dev, event)
    t_crit = t_dist.ppf(0.975, df=n_hets - 1)
    cf_lower = phased_dev_to_cf(mean_dev - t_crit * se_dev, event)
    cf_upper = phased_dev_to_cf(mean_dev + t_crit * se_dev, event)
    
    return {
        'n_hets_inside': n_hets,
        'mean_phased_dev': mean_dev,
        'se_phased_dev': se_dev,
        'cf_estimate': cf,
        'cf_lower_95': cf_lower,
        'cf_upper_95': cf_upper,
        't_stat': float(t_stat),
        'p_onesample': p_onesided,
        'p_twosided': float(p_two),
        'p_permutation': p_permutation,
        'significant_raw': p_onesided < p_threshold,
    }


### Main pipeline

In [ ]:
def run_per_sample_pipeline(manifest_csv, simulated_dir, output_csv,
                            known_germline_chroms=None, het_lo=0.02, het_hi=0.98,
                            min_hets=3, p_threshold=0.05, n_permutations=10000,
                            perm_seed=42, debug=True):
    """
    For each row in the manifest: load sample, use ground-truth coords,
    use own Phased_Haplotype, test detection.
    
    Parameters
    ----------
    manifest_csv : str - path to simulation_manifest.csv
    simulated_dir : str - path to directory containing simulated SNP files
    output_csv : str - path for output results CSV
    known_germline_chroms : dict or None - {control: [chroms]} to skip
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    p_threshold : float - significance threshold
    n_permutations : int - number of sign-flip permutations
    perm_seed : int - random seed for permutation test
    debug : bool - print progress
    """
    if known_germline_chroms is None:
        known_germline_chroms = {}
    
    manifest = pd.read_csv(manifest_csv)
    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    
    print(f"Manifest: {len(manifest)} samples")
    
    all_results = []
    n_skipped_germline = 0
    n_skipped_no_file = 0
    n_skipped_insufficient = 0
    n_tested = 0
    
    for i, row in manifest.iterrows():
        control = row['Sample']
        chrom = row['Chromosome']
        event_raw = row['Type']
        event = event_map.get(event_raw, event_raw)
        cf_true = row['Fraction_Percent']
        start_bp = int(row['Start_bp'])
        end_bp = int(row['End_bp'])
        size = row['Length_Category']
        geometry = row['Geometry']
        
        # Skip germline chromosomes
        if control in known_germline_chroms and chrom in known_germline_chroms[control]:
            n_skipped_germline += 1
            continue
        
        # Derive SNP filename
        snp_name = row['File_Name'].replace(
            '_PON_normalised_read_depths_and_LRR.txt', '_SNPs.txt'
        )
        snp_path = os.path.join(simulated_dir, snp_name)
        
        if not os.path.exists(snp_path):
            n_skipped_no_file += 1
            continue
        
        try:
            snp_data = pd.read_csv(snp_path, sep='\t')
        except Exception:
            n_skipped_no_file += 1
            continue
        
        # Analyse this sample — pass all config explicitly
        result = analyse_single_sample(
            snp_data, chrom, start_bp, end_bp, event,
            het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
            p_threshold=p_threshold, n_permutations=n_permutations,
            perm_seed=perm_seed
        )
        
        if result is None:
            n_skipped_insufficient += 1
            all_results.append({
                'control': control,
                'event': event,
                'chromosome': chrom,
                'start_pos': start_bp,
                'end_pos': end_bp,
                'cf_true': cf_true,
                'size': size,
                'geometry': geometry,
                'sample_name': snp_name.replace('_SNPs.txt', ''),
                'status': 'insufficient_hets',
                'n_hets_inside': 0,
                'mean_phased_dev': np.nan,
                'se_phased_dev': np.nan,
                'cf_estimate': np.nan,
                'cf_lower_95': np.nan,
                'cf_upper_95': np.nan,
                't_stat': np.nan,
                'p_onesample': np.nan,
                'p_twosided': np.nan,
                'p_permutation': np.nan,
                'significant_raw': False,
            })
            continue
        
        result['control'] = control
        result['event'] = event
        result['chromosome'] = chrom
        result['start_pos'] = start_bp
        result['end_pos'] = end_bp
        result['cf_true'] = cf_true
        result['size'] = size
        result['geometry'] = geometry
        result['sample_name'] = snp_name.replace('_SNPs.txt', '')
        result['status'] = 'tested'
        all_results.append(result)
        n_tested += 1
        
        if (i + 1) % 500 == 0 and debug:
            print(f"  Processed {i+1}/{len(manifest)}...")
    
    results_df = pd.DataFrame(all_results)
    
    # Save
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    results_df.to_csv(output_csv, index=False)
    print(f"\n✅ Saved {len(results_df)} results to {output_csv}")
    
    # ── Summary ──
    print(f"\n{'='*70}")
    print(f"Per-Sample Phased Analysis — Ground-Truth Ceiling")
    print(f"{'='*70}")
    print(f"  Tested:                 {n_tested}")
    print(f"  Skipped (germline):     {n_skipped_germline}")
    print(f"  Skipped (no file):      {n_skipped_no_file}")
    print(f"  Skipped (insuf. hets):  {n_skipped_insufficient}")
    
    tested = results_df[results_df['status'] == 'tested']
    
    # Overall sensitivity by CF
    print(f"\n  {'CF_true':<10} {'N':>5} {'Detected':>10} {'Sensitivity':>12} {'Mean_CF_est':>12} {'Mean_p':>12}")
    print(f"  {'-'*65}")
    for cf in sorted(tested['cf_true'].unique()):
        subset = tested[tested['cf_true'] == cf]
        n = len(subset)
        n_sig = subset['significant_raw'].sum()
        mean_est = subset.loc[subset['significant_raw'], 'cf_estimate'].mean()
        mean_p = subset['p_onesample'].mean()
        est_str = f'{mean_est:.5f}' if n_sig > 0 else 'N/A'
        print(f"  {cf:<10.4f} {n:>5} {n_sig:>5}/{n:<4} {n_sig/n*100:>10.1f}% {est_str:>12} {mean_p:>12.2e}")
    
    # By event type
    print(f"\n  By event type:")
    for evt in ['CN-LOH', 'GAIN', 'LOSS']:
        evt_df = tested[tested['event'] == evt]
        if len(evt_df) == 0:
            continue
        print(f"\n    {evt}:")
        for cf in sorted(evt_df['cf_true'].unique()):
            subset = evt_df[evt_df['cf_true'] == cf]
            n = len(subset)
            n_sig = subset['significant_raw'].sum()
            print(f"      CF={cf:.4f}: {n_sig:>4}/{n:<4} ({n_sig/n*100:5.1f}%)")
    
    # By size
    print(f"\n  By size:")
    for size in ['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']:
        size_df = tested[tested['size'] == size]
        if len(size_df) == 0:
            continue
        print(f"\n    {size}:")
        for cf in sorted(size_df['cf_true'].unique()):
            subset = size_df[size_df['cf_true'] == cf]
            n = len(subset)
            n_sig = subset['significant_raw'].sum()
            print(f"      CF={cf:.4f}: {n_sig:>4}/{n:<4} ({n_sig/n*100:5.1f}%)")
    
    return results_df


In [ ]:
# Run per-sample analysis
per_sample_results = run_per_sample_pipeline(
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    output_csv=OUTPUT_CSV,
    known_germline_chroms=KNOWN_GERMLINE_CHROMS,
    het_lo=HET_LO,
    het_hi=HET_HI,
    min_hets=MIN_HETS,
    p_threshold=P_THRESHOLD,
    n_permutations=N_PERMUTATIONS,
    perm_seed=PERM_SEED,
    debug=DEBUG
)


In [ ]:
def plot_known_region_phased(member_snps, chrom, region_start, region_end, event,
                              phase_lookup,
                              cf_result=None, sample_name='',
                              het_lo=0.02, het_hi=0.98,
                              chromosome_sizes=None, ideogram_file=None,
                              save_path=None):
    """
    Plot a chromosome with phased SNPs colored by haplotype in the known region.
    
    Parameters
    ----------
    member_snps : DataFrame
        Full SNP data for this sample (all chromosomes OK, will be filtered).
        Must have columns: chromosome, position, VAF, and optionally lrr_filt.
    chrom : str
    region_start, region_end : int
        Known mCA region boundaries.
    event : str
        'CN-LOH', 'GAIN', 'LOSS'
    phase_lookup : dict
        {position: Phased_Haplotype} from reference sample.
    favored_haplotype : int (0 or 1)
    cf_result : dict or None
        Output from estimate_cf_single_region (has cf_estimate, p_onesample, etc.)
    sample_name : str
    het_lo, het_hi : float
    chromosome_sizes : dict
    ideogram_file : str or None
    save_path : str or None
    
    Returns
    -------
    matplotlib Figure
    """
    
    # ── Event-specific colors ──
    hap_colors = {
        'GAIN':    {'plus': red4,  'minus': red2},
        'LOSS':    {'plus': blue4, 'minus': blue2},
        'CN-LOH':  {'plus': orange3, 'minus': orange2},
    }
    default_colors = {'plus': grey5, 'minus': grey2}
    colors = hap_colors.get(event, default_colors)
    color_plus = colors['plus']    # Haplotype A (favored)
    color_minus = colors['minus']  # Haplotype B
    
    event_shading = {
        'GAIN': red1, 'LOSS': blue1, 'CN-LOH': orange1,
    }
    shade_color = event_shading.get(event, grey1)
    
    event_border = {
        'GAIN': red3, 'LOSS': blue3, 'CN-LOH': orange3,
    }
    border_color = event_border.get(event, grey3)
    
    # ── Prepare data ──
    d = member_snps[member_snps['chromosome'].astype(str) == str(chrom)].copy()
    d = d.sort_values('position').reset_index(drop=True)
    
    if d.empty:
        print(f"  No data for {chrom}")
        return None
    
    # Identify hets
    plot_het_lo = cf_result.get('eff_het_lo', het_lo) if cf_result else het_lo
    plot_het_hi = cf_result.get('eff_het_hi', het_hi) if cf_result else het_hi
    d['is_het'] = (d['VAF'] >= plot_het_lo) & (d['VAF'] <= plot_het_hi)
    
    # Compute AI for hets
    d['ai'] = np.nan
    het_mask = d['is_het']
    d.loc[het_mask, 'ai'] = np.abs(d.loc[het_mask, 'VAF'] - 0.5)
    
    # Compute phased_dev for hets in the phase map
    d['phased_dev'] = np.nan
    d['haplotype'] = np.nan  # 1 = favored (A), 0 = other (B)

    # Get effective het bounds from cf_result (reflects which pass was used)
    plot_het_lo = cf_result.get('eff_het_lo', het_lo) if cf_result else het_lo
    plot_het_hi = cf_result.get('eff_het_hi', het_hi) if cf_result else het_hi

    for idx, row in d.iterrows():
        pos = int(row['position'])
        if pos in phase_lookup:
            if not (plot_het_lo <= row['VAF'] <= plot_het_hi):
                continue
            phase = phase_lookup[pos]
            dev = (2 * phase - 1) * (row['VAF'] - 0.5)
            d.at[idx, 'phased_dev'] = dev
            d.at[idx, 'haplotype'] = phase
    
    # Separate subsets
    hets = d[d['is_het']].copy()
    homs = d[~d['is_het']].copy()
    
    # Region masks
    in_region = (d['position'] >= region_start) & (d['position'] <= region_end)
    het_in_region = d['is_het'] & in_region
    het_outside = d['is_het'] & ~in_region
    
    # Phased hets in region
    phased_in_region = in_region & d['phased_dev'].notna()
    hap_A = phased_in_region & (d['haplotype'] == 1.0)
    hap_B = phased_in_region & (d['haplotype'] == 0.0)
    unphased_het_in_region = in_region & d['phased_dev'].isna() & d['is_het']
    
    # ── Significance info ──
    is_significant = False
    cf_text = ''
    p_text = ''
    if cf_result is not None:
        is_significant = cf_result.get('significant_raw', False)
        cf_est = cf_result.get('cf_estimate', 0)
        cf_lo  = cf_result.get('cf_lower_95', None)
        cf_hi  = cf_result.get('cf_upper_95', None)
        p_val = cf_result.get('p_onesample', 1)
        cf_text = f'CF={cf_est:.3f} ({cf_lo:.3f}–{cf_hi:.3f})' if cf_lo is not None else f'CF={cf_est:.3f}'
        p_text = f'p={p_val:.2e}'
    
    # ── Determine if LRR data exists ──
    has_lrr = 'lrr_filt' in d.columns and d['lrr_filt'].notna().sum() > 10
    
    # ── Build figure ──
    if has_lrr:
        n_panels = 5
        height_ratios = [4, 6, 3, 3, 0.6]
        fig_height = 4.5
    else:
        n_panels = 4
        height_ratios = [6, 3, 0.6]
        fig_height = 3.0
    
    fig, axes = plt.subplots(
        n_panels, 1, figsize=(14, fig_height), sharex=False,
        gridspec_kw={'height_ratios': height_ratios, 'hspace': 0.08},
        constrained_layout=True
    )
    
    panel_idx = 0
    
    # ═══ PANEL: LRR (if available) ═══
    if has_lrr:
        ax_lrr = axes[panel_idx]
        panel_idx += 1
        
        # Background: all SNPs grey
        ax_lrr.scatter(d['position'], d['lrr_filt'], s=6, color=grey3, alpha=0.3, zorder=1)
        
        # Inside region: color by haplotype
        if hap_A.any():
            ax_lrr.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'lrr_filt'],
                          s=10, color=color_plus, alpha=0.8, zorder=10)
        if hap_B.any():
            ax_lrr.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'lrr_filt'],
                          s=10, color=color_minus, alpha=0.8, zorder=10)
        if unphased_het_in_region.any():
            ax_lrr.scatter(d.loc[unphased_het_in_region, 'position'], 
                          d.loc[unphased_het_in_region, 'lrr_filt'],
                          s=8, color=grey4, alpha=0.5, zorder=5)
        
        ax_lrr.axhline(0, linestyle='--', color=grey2, zorder=0)
        ax_lrr.set_ylabel('LRR')
        ax_lrr.set_ylim(-1.5, 1.5)
    
    # ═══ PANEL: BAF ═══
    ax_baf = axes[panel_idx]
    panel_idx += 1
    
    # Homozygous: light grey background
    if not homs.empty:
        ax_baf.scatter(homs['position'], homs['VAF'], s=6, color='lightgray', alpha=0.6, zorder=1)
    
    # Hets outside region: grey
    if het_outside.any():
        ax_baf.scatter(d.loc[het_outside, 'position'], d.loc[het_outside, 'VAF'],
                      s=8, color=grey4, alpha=0.8, zorder=5)
    
    # Hets inside region: color by haplotype
    if hap_A.any():
        ax_baf.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'VAF'],
                      s=14, color=color_plus, alpha=0.9, zorder=10)
    if hap_B.any():
        ax_baf.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'VAF'],
                      s=14, color=color_minus, alpha=0.9, zorder=10)
    if unphased_het_in_region.any():
        ax_baf.scatter(d.loc[unphased_het_in_region, 'position'],
                      d.loc[unphased_het_in_region, 'VAF'],
                      s=10, color=grey4, alpha=0.5, zorder=5)
    
    ax_baf.axhline(0.5, linestyle='--', color=grey2, zorder=0)
    ax_baf.set_ylabel('BAF')
    ax_baf.set_ylim(-0.05, 1.05)
    
    # ═══ PANEL: AI (unsigned) ═══
    ax_ai = axes[panel_idx]
    panel_idx += 1
    
    # Outside region: grey
    if het_outside.any():
        ax_ai.scatter(d.loc[het_outside, 'position'], d.loc[het_outside, 'ai'],
                     s=8, color=grey4, alpha=0.6, zorder=5)
    
    # Inside region: color by haplotype
    if hap_A.any():
        ax_ai.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'ai'],
                     s=14, color=color_plus, alpha=0.9, zorder=10)
    if hap_B.any():
        ax_ai.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'ai'],
                     s=14, color=color_minus, alpha=0.9, zorder=10)
    if unphased_het_in_region.any():
        ax_ai.scatter(d.loc[unphased_het_in_region, 'position'],
                     d.loc[unphased_het_in_region, 'ai'],
                     s=10, color=grey4, alpha=0.5, zorder=5)
    
    ax_ai.axhline(0.03, linestyle='--', color=grey2, zorder=0)
    ax_ai.set_ylabel('AI')
    
    # ═══ PANEL: Phased deviation ═══
    ax_phased = axes[panel_idx]
    panel_idx += 1
    
    # Only phased SNPs in the region have phased_dev
    phased_hets = d[d['phased_dev'].notna()].copy()
    if not phased_hets.empty:
        pos_dev = phased_hets[phased_hets['haplotype'] == 1.0]
        neg_dev = phased_hets[phased_hets['haplotype'] == 0.0]
        
        if not pos_dev.empty:
            ax_phased.scatter(pos_dev['position'], pos_dev['phased_dev'],
                            s=14, color=color_plus, alpha=0.9, zorder=10)
        if not neg_dev.empty:
            ax_phased.scatter(neg_dev['position'], neg_dev['phased_dev'],
                            s=14, color=color_minus, alpha=0.9, zorder=10)
    
    ax_phased.axhline(0, linestyle='--', color=grey2, zorder=0)
    ax_phased.set_ylabel('Phased\ndeviation')
    
    # Add mean line if we have a result
    if cf_result is not None and phased_hets.shape[0] > 0:
        mean_dev = cf_result.get('mean_phased_dev', 0)
        line_color = color_plus if is_significant else grey3
        line_style = '-' if is_significant else ':'
        ax_phased.axhline(mean_dev, linestyle=line_style, color=line_color, 
                         linewidth=2, alpha=0.7, zorder=15,
                         xmin=(region_start - d['position'].min()) / (d['position'].max() - d['position'].min()),
                         xmax=(region_end - d['position'].min()) / (d['position'].max() - d['position'].min()))
    
    # ═══ PANEL: Ideogram ═══
    ax_ideo = axes[panel_idx]
    
    try:
        plot_chromosome(ideogram_file, chrom, ax_ideo)
    except Exception:
        ax_ideo.set_xlim(0, chromosome_sizes.get(chrom, d['position'].max()) if chromosome_sizes else d['position'].max())
        ax_ideo.set_yticks([])
        ax_ideo.text(0.5, 0.5, chrom, transform=ax_ideo.transAxes, ha='center', va='center', fontsize=12)




    # --- PANEL 5: Ideogram ---
    plot_chromosome(ideogram_file, chrom, ax_ideo)
    
    # X-limits
    chromosome_size = chromosome_sizes[chrom]
    chrom_size_mb = chromosome_size / 1e6
    # ax_ideo.text(1.01, 0.5, f"{chrom_size_mb:.0f} Mb", transform=ax_ideo.transAxes, 
            #  ha="left", va="center", fontsize=12, color='black')
    ax_ideo.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    
    # ═══ Region shading + dashed outline on ALL panels ═══
    data_axes = [ax for ax in axes[:-1]]  # all except ideogram
    
    for ax in data_axes:
        # Shaded background
        ax.axvspan(region_start, region_end, color=shade_color, alpha=0.25, zorder=0)
        
        # Dashed border
        ymin, ymax = ax.get_ylim()
        rect = mpatches.Rectangle(
            (region_start, ymin), region_end - region_start, ymax - ymin,
            linewidth=1.5, edgecolor=border_color, facecolor='none',
            linestyle='--', zorder=20
        )
        ax.add_patch(rect)
    
    # ═══ Spine styling ═══
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['left'].set_color(grey3)
        ax.spines['bottom'].set_color(grey3)
    
    # ═══ X-limits ═══
    if chromosome_sizes and chrom in chromosome_sizes:
        chrom_size = chromosome_sizes[chrom]
    else:
        chrom_size = int(d['position'].max() * 1.05)
    
    for ax in axes:
        ax.set_xlim(0, chrom_size)
    
    # Hide x ticks on upper panels
    for ax in axes[:-1]:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    # axes[-1].tick_params(axis='x', bottom=True, labelbottom=True)
    
    # ═══ Title ═══
    sig_str = '✓ SIGNIFICANT' if is_significant else '✗ not significant'
    title = f'{sample_name}\n{chrom}: {event} {region_start/1e6:.1f}–{region_end/1e6:.1f} Mb'
    if cf_result is not None:
        title += f'  |  {cf_text}  |  {p_text}  |  {sig_str}'
        n_phased = cf_result.get('n_hets_inside', 0)
        title += f'  |  n={n_phased} phased hets'
    
    # ═══ Legend ═══
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_plus,
               markersize=8, label=f'Haplotype A ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_minus,
               markersize=8, label=f'Haplotype B ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=grey4,
               markersize=6, label='Unphased'),
        mpatches.Patch(facecolor=shade_color, edgecolor=border_color,
                      linestyle='--', label='Known region'),
    ]
    fig.suptitle(title, fontsize=9, fontweight='bold', x=0.02, ha='left')
    fig.legend(handles=legend_elements, fontsize=7, ncol=4,
               loc='upper right', bbox_to_anchor=(0.99, 0.99),
               framealpha=0.9, edgecolor='lightgrey')
    
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    
    return fig

def plot_all_per_sample_results(results_df, chromosome_sizes, ideogram_file=None,
                                 het_lo=0.02, het_hi=0.98,
                                 simulated_dir='CNV_panel_simulated_Samples_Feb2026/Simulated_samples_TEST_set',
                                 output_dir='known_region_plots',
                                 plot_all=True, max_plots=None):
    """
    Generate phased plots for per-sample results (ground-truth ceiling analysis).
    
    Each sample uses its OWN Phased_Haplotype for coloring — no family grouping,
    no reference sample, no cross-sample phasing.
    
    Parameters
    ----------
    results_df : DataFrame
        Output from run_per_sample_pipeline. Must have columns:
        sample_name, chromosome, start_pos, end_pos, event, cf_true,
        significant_raw, cf_estimate, cf_lower_95, cf_upper_95,
        p_onesample, mean_phased_dev, n_hets_inside.
    chromosome_sizes : dict
    ideogram_file : str or None
    simulated_dir : str
        Path to directory containing simulated sample files.
    plot_all : bool
        If True, plot all results. If False, only significant + interesting CFs.
    max_plots : int or None
        Safety limit on total plots generated.
    """
    import matplotlib.pyplot as plt
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Only plot tested samples (skip insufficient_hets)
    plotable = results_df[results_df['status'] == 'tested'].copy()
    
    n_plotted = 0
    n_total = len(plotable)
    
    for _, row in plotable.iterrows():
        if max_plots and n_plotted >= max_plots:
            print(f"  Reached max_plots={max_plots}, stopping")
            return
        
        # Skip non-significant unless plotting all or at interesting CFs
        if not plot_all and not row['significant_raw']:
            if row['cf_true'] not in [0.005, 0.01, 0.05]:
                continue
        
        sample_name = row['sample_name']
        chrom = row['chromosome']
        event = row['event']
        region_start = int(row['start_pos'])
        region_end = int(row['end_pos'])
        
        # ── Load SNP file ──
        snp_file = os.path.join(simulated_dir, sample_name + '_SNPs.txt')
        if not os.path.exists(snp_file):
            continue
        
        try:
            member_snps = pd.read_csv(snp_file, sep='\t')
        except Exception:
            continue
        
        if 'Phased_Haplotype' not in member_snps.columns:
            continue
        
        # ── Load and merge LRR if available ──
        lrr_file = os.path.join(simulated_dir, sample_name + '_PON_normalised_read_depths_and_LRR.txt')
        if os.path.exists(lrr_file):
            try:
                lrr_data = pd.read_csv(lrr_file, sep='\t')
                if 'LRR' in lrr_data.columns:
                    lrr_cols = ['chromosome', 'position', 'LRR']
                    if 'lrr_filt' in lrr_data.columns:
                        lrr_cols.append('lrr_filt')
                    member_snps = member_snps.merge(
                        lrr_data[lrr_cols].rename(columns={'LRR': 'lrr_raw'}),
                        on=['chromosome', 'position'], how='left')
                    if 'lrr_filt' not in member_snps.columns:
                        member_snps['lrr_filt'] = member_snps['lrr_raw']
            except Exception:
                pass
        
        # ── Build phase_lookup from this sample's OWN Phased_Haplotype ──
        chrom_snps = member_snps[member_snps['chromosome'].astype(str) == str(chrom)]
        region_snps = chrom_snps[
            (chrom_snps['position'] >= region_start) & 
            (chrom_snps['position'] <= region_end)
        ]
        phased_snps = region_snps[region_snps['Phased_Haplotype'].notna()]
        
        if len(phased_snps) < 3:
            continue
        
        phase_lookup = dict(zip(
            phased_snps['position'].astype(int),
            phased_snps['Phased_Haplotype'].astype(float)
        ))
        
        # ── Build cf_result dict for the plot function ──
        cf_result = {
            'cf_estimate': row['cf_estimate'],
            'cf_lower_95': row['cf_lower_95'],
            'cf_upper_95': row['cf_upper_95'],
            'p_onesample': row['p_onesample'],
            'significant_raw': row['significant_raw'],
            'mean_phased_dev': row['mean_phased_dev'],
            'n_hets_inside': row['n_hets_inside'],
        }
        
        # ── Filename ──
        sig_tag = 'sig' if row['significant_raw'] else 'ns'
        cf_pct = row['cf_true'] * 100
        filename = f"{sample_name}_{chrom}_{sig_tag}.png"
        save_path = os.path.join(output_dir, filename)
        
        # ── Plot ──
        plot_known_region_phased(
            member_snps, chrom, region_start, region_end, event,
            phase_lookup, favored_haplotype,
            cf_result=cf_result, sample_name=sample_name,
            het_lo=het_lo, het_hi=het_hi,
            chromosome_sizes=chromosome_sizes, ideogram_file=ideogram_file,
            save_path=save_path
        )
        
        n_plotted += 1
        if n_plotted % 50 == 0:
            print(f"  Plotted {n_plotted} / {n_total}...")
    
    print(f"  Done: {n_plotted} plots saved to {output_dir}/")

In [ ]:
def plot_known_region_smaller_plot(member_snps, chrom, region_start, region_end, event,
                              phase_lookup,
                              cf_result=None, sample_name='',
                              het_lo=0.02, het_hi=0.98,
                              chromosome_sizes=None, ideogram_file=None,
                              save_path=None):
    """
    Plot a chromosome with phased SNPs colored by haplotype in the known region.
    AI panel removed. Compact layout: BAF (2x height) + phased deviation (1x) + ideogram.
    """

    # ── Event-specific colors ──
    hap_colors = {
        'GAIN':   {'plus': red4,    'minus': red2},
        'LOSS':   {'plus': blue4,   'minus': blue2},
        'CN-LOH': {'plus': orange3, 'minus': orange2},
    }
    default_colors = {'plus': grey5, 'minus': grey2}
    colors     = hap_colors.get(event, default_colors)
    color_plus  = colors['plus']
    color_minus = colors['minus']

    event_shading = {'GAIN': red1,    'LOSS': blue1,   'CN-LOH': orange1}
    event_border  = {'GAIN': red3,    'LOSS': blue3,   'CN-LOH': orange3}
    shade_color  = event_shading.get(event, grey1)
    border_color = event_border.get(event, grey3)

    # ── Prepare data ──
    d = member_snps[member_snps['chromosome'].astype(str) == str(chrom)].copy()
    d = d.sort_values('position').reset_index(drop=True)
    if d.empty:
        print(f"  No data for {chrom}")
        return None

    plot_het_lo = cf_result.get('eff_het_lo', het_lo) if cf_result else het_lo
    plot_het_hi = cf_result.get('eff_het_hi', het_hi) if cf_result else het_hi
    d['is_het'] = (d['VAF'] >= plot_het_lo) & (d['VAF'] <= plot_het_hi)
    d['phased_dev'] = np.nan
    d['haplotype']  = np.nan

    for idx, row in d.iterrows():
        pos = int(row['position'])
        if pos in phase_lookup:
            if not (plot_het_lo <= row['VAF'] <= plot_het_hi):
                continue
            phase = phase_lookup[pos]
            d.at[idx, 'phased_dev'] = (2 * phase - 1) * (row['VAF'] - 0.5)
            d.at[idx, 'haplotype']  = phase

    homs           = d[~d['is_het']].copy()
    in_region      = (d['position'] >= region_start) & (d['position'] <= region_end)
    het_outside    = d['is_het'] & ~in_region
    phased_in_region       = in_region & d['phased_dev'].notna()
    hap_A                  = phased_in_region & (d['haplotype'] == 1.0)
    hap_B                  = phased_in_region & (d['haplotype'] == 0.0)
    unphased_het_in_region = in_region & d['phased_dev'].isna() & d['is_het']

    # ── Significance info ──
    is_significant = False
    cf_text = ''
    p_text  = ''
    if cf_result is not None:
        is_significant = cf_result.get('significant_raw', False)
        cf_est = cf_result.get('cf_estimate', 0)
        cf_lo  = cf_result.get('cf_lower_95', None)
        cf_hi  = cf_result.get('cf_upper_95', None)
        p_val  = cf_result.get('p_onesample', 1)
        cf_text = f'CF={cf_est:.3f} ({cf_lo:.3f}–{cf_hi:.3f})' if cf_lo is not None else f'CF={cf_est:.3f}'
        p_text  = f'p={p_val:.2e}'

    # ── LRR availability ──
    has_lrr = 'lrr_filt' in d.columns and d['lrr_filt'].notna().sum() > 10

    # ── Figure layout ──
    # Without LRR: [BAF(4), phased(2), ideogram(0.75)]  → fig_height ~2.7
    # With    LRR: [LRR(3), BAF(4), phased(2), ideogram(0.75)] → fig_height ~3.3
    if has_lrr:
        height_ratios = [3, 4, 2, 0.6]
        fig_height    = 2.8
        n_panels      = 4
    else:
        height_ratios = [4, 2, 0.6]
        fig_height    = 2.2
        n_panels      = 3

    fig, axes = plt.subplots(
        n_panels, 1, figsize=(14, fig_height), sharex=False,
        gridspec_kw={'height_ratios': height_ratios, 'hspace': 0.06},
        constrained_layout=True
    )

    panel_idx = 0

    # ═══ PANEL: LRR (optional) ═══
    if has_lrr:
        ax_lrr = axes[panel_idx]; panel_idx += 1
        ax_lrr.scatter(d['position'], d['lrr_filt'], s=4, color=grey3, alpha=0.3, zorder=1)
        if hap_A.any():
            ax_lrr.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'lrr_filt'],
                           s=8, color=color_plus, alpha=0.8, zorder=10)
        if hap_B.any():
            ax_lrr.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'lrr_filt'],
                           s=8, color=color_minus, alpha=0.8, zorder=10)
        if unphased_het_in_region.any():
            ax_lrr.scatter(d.loc[unphased_het_in_region, 'position'],
                           d.loc[unphased_het_in_region, 'lrr_filt'],
                           s=6, color=grey4, alpha=0.5, zorder=5)
        ax_lrr.axhline(0, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
        ax_lrr.set_ylabel('LRR', fontsize=8)
        ax_lrr.set_ylim(-1.5, 1.5)

    # ═══ PANEL: BAF ═══
    ax_baf = axes[panel_idx]; panel_idx += 1
    if not homs.empty:
        ax_baf.scatter(homs['position'], homs['VAF'], s=4, color='lightgray', alpha=0.6, zorder=1)
    if het_outside.any():
        ax_baf.scatter(d.loc[het_outside, 'position'], d.loc[het_outside, 'VAF'],
                       s=6, color=grey4, alpha=0.8, zorder=5)
    if hap_A.any():
        ax_baf.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'VAF'],
                       s=10, color=color_plus, alpha=0.9, zorder=10)
    if hap_B.any():
        ax_baf.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'VAF'],
                       s=10, color=color_minus, alpha=0.9, zorder=10)
    if unphased_het_in_region.any():
        ax_baf.scatter(d.loc[unphased_het_in_region, 'position'],
                       d.loc[unphased_het_in_region, 'VAF'],
                       s=8, color=grey4, alpha=0.5, zorder=5)
    ax_baf.axhline(0.5, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
    ax_baf.set_ylabel('BAF', fontsize=8)
    ax_baf.set_ylim(-0.05, 1.05)

    # ═══ PANEL: Phased deviation ═══
    ax_phased = axes[panel_idx]; panel_idx += 1
    phased_hets = d[d['phased_dev'].notna()].copy()
    if not phased_hets.empty:
        pos_dev = phased_hets[phased_hets['haplotype'] == 1.0]
        neg_dev = phased_hets[phased_hets['haplotype'] == 0.0]
        if not pos_dev.empty:
            ax_phased.scatter(pos_dev['position'], pos_dev['phased_dev'],
                              s=10, color=color_plus, alpha=0.9, zorder=10)
        if not neg_dev.empty:
            ax_phased.scatter(neg_dev['position'], neg_dev['phased_dev'],
                              s=10, color=color_minus, alpha=0.9, zorder=10)
    ax_phased.axhline(0, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
    ax_phased.set_ylabel('Phased\ndev.', fontsize=8)

    if cf_result is not None and not phased_hets.empty:
        mean_dev   = cf_result.get('mean_phased_dev', 0)
        line_color = color_plus if is_significant else grey3
        line_style = '-' if is_significant else ':'
        pos_range  = d['position'].max() - d['position'].min()
        ax_phased.axhline(
            mean_dev, linestyle=line_style, color=line_color,
            linewidth=1.5, alpha=0.8, zorder=15,
            xmin=(region_start - d['position'].min()) / pos_range,
            xmax=(region_end   - d['position'].min()) / pos_range,
        )

    # ═══ PANEL: Ideogram ═══
    ax_ideo = axes[panel_idx]
    try:
        plot_chromosome(ideogram_file, chrom, ax_ideo)
    except Exception:
        ax_ideo.set_xlim(0, chromosome_sizes.get(chrom, d['position'].max()) if chromosome_sizes else d['position'].max())
        ax_ideo.set_yticks([])
        # ax_ideo.text(0.5, 0.5, chrom, transform=ax_ideo.transAxes,
        #              ha='center', va='center', fontsize=10)

    if chromosome_sizes and chrom in chromosome_sizes:
        chrom_size_mb = chromosome_sizes[chrom] / 1e6
        # ax_ideo.text(1.01, 0.5, f"{chrom_size_mb:.0f} Mb",
                    #  transform=ax_ideo.transAxes, ha='left', va='center',
                    #  fontsize=9, color='black')
        ax_ideo.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # ═══ Region shading + dashed outline ═══
    data_axes = [ax for ax in axes[:-1]]
    for ax in data_axes:
        ax.axvspan(region_start, region_end, color=shade_color, alpha=0.25, zorder=0)
        ymin, ymax = ax.get_ylim()
        rect = mpatches.Rectangle(
            (region_start, ymin), region_end - region_start, ymax - ymin,
            linewidth=1.2, edgecolor=border_color, facecolor='none',
            linestyle='--', zorder=20
        )
        ax.add_patch(rect)

    # ═══ Spine styling ═══
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.2)
        ax.spines['bottom'].set_linewidth(1.2)
        ax.spines['left'].set_color(grey3)
        ax.spines['bottom'].set_color(grey3)
        ax.tick_params(labelsize=7)

    # ═══ X-limits ═══
    chrom_size = chromosome_sizes.get(chrom, int(d['position'].max() * 1.05)) \
                 if chromosome_sizes else int(d['position'].max() * 1.05)
    for ax in axes:
        ax.set_xlim(0, chrom_size)

    for ax in axes[:-1]:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    # axes[-1].tick_params(axis='x', bottom=True, labelbottom=True)

    # ═══ Title ═══
    sig_str = '✓ SIGNIFICANT' if is_significant else '✗ not significant'
    title = f'{sample_name}\n{chrom}: {event} {region_start/1e6:.1f}–{region_end/1e6:.1f} Mb'
    if cf_result is not None:
        n_phased = cf_result.get('n_hets_inside', 0)
        title += f'  |  {cf_text}  |  {p_text}  |  {sig_str}  |  n={n_phased} phased hets'

    # ═══ Legend ═══
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_plus,
               markersize=7, label=f'Haplotype A ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_minus,
               markersize=7, label=f'Haplotype B ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=grey4,
               markersize=5, label='Unphased'),
        mpatches.Patch(facecolor=shade_color, edgecolor=border_color,
                       linestyle='--', label='Known region'),
    ]
    fig.suptitle(title, fontsize=9, fontweight='bold', x=0.02, ha='left')
    fig.legend(handles=legend_elements, fontsize=7, ncol=4,
               loc='upper right', bbox_to_anchor=(0.99, 0.99),
               framealpha=0.9, edgecolor='lightgrey')

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

    return fig

In [ ]:
plot_all_per_sample_results(
    per_sample_results,
    chromosome_sizes=chromosome_sizes,
    ideogram_file=ideogram_file,
    simulated_dir='CNV_panel_simulated_Samples_Feb2026/Simulated_samples_TEST_set',
    output_dir=os.path.join(OUTPUT_DIR, 'Phased_per_sample_plots'),
    plot_all=False,
    max_plots=100
)


# 2. Simulated samples: Using high CF sample phasing & coordinates to call mCA cell fraction in lower CF sample ('longitudinal analysis')

- Tests multiple reference CFs (100%, 75%, 50%, 25%) independently per family, showing how phasing quality degrades with lower reference CF.

For each simulated mCA family:
  1. Check if the mCA was detected (TP) by the unphased caller at a reference CF
  2. If yes, derive phasing from the reference sample's VAF (real-world workflow)
  3. Apply that phasing to all lower-CF samples in the family
  4. Test whether the mCA is detectable at each lower CF

Requires:
  - truth_comparison.csv (unphased caller results with detected coordinates)
  - simulation_manifest.csv
  - Simulated SNP files

### Configuration

In [ ]:
# Reference CFs to test phasing from (highest priority first)
REFERENCE_CFS = [1.0, 0.75, 0.5, 0.25]

# Target CFs to test detection at
fractions = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]
TARGET_CFS = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]

# Minimum median AI in reference to accept phasing
MIN_MEDIAN_AI = 0.01


In [ ]:
# PON configuration
PON_BASE_PATH = "."  # adjust if notebook isn't in the same folder
PON_LIBRARIES = ['SLX_19285', 'SLX_20125', 'SLX_20127']

# Read PON sample list from master matrix
_pon_matrix = pd.read_csv(
    'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv',
    sep='\t', nrows=0
)
FINAL_TIMEPOINT_CONTROLS = [c for c in _pon_matrix.columns if c.startswith('CNTRL')]

# Baseline weight (set after sweep)
BASELINE_WEIGHT = 0.75  # update once sweep complete

# Known germline mCA chromosomes — skip these
KNOWN_GERMLINE_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

### Core functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SHARED FUNCTIONS — used by BOTH longitudinal pipeline and FPR pipeline
# ═══════════════════════════════════════════════════════════════════════════
# Cell 21: Replace your current cell 21 with this.
# These are the ONLY definitions of these functions in the notebook.
# ═══════════════════════════════════════════════════════════════════════════


def build_population_baselines(control_snp_files, het_lo=0.01, het_hi=0.99):
    """
    Compute per-position median VAF across all control samples.
    Removes systematic probe capture bias.

    Parameters
    ----------
    control_snp_files : dict {control_name: filepath_or_DataFrame}
    het_lo, het_hi : float - exclude clear homozygotes

    Returns
    -------
    baselines : dict {(chrom, position): median_vaf}
    """
    from collections import defaultdict

    pos_vafs = defaultdict(list)

    for ctrl_name, snp_source in control_snp_files.items():
        if isinstance(snp_source, str):
            try:
                snps = pd.read_csv(snp_source, sep='\t')
            except Exception as e:
                print(f"  Warning: could not load {ctrl_name}: {e}")
                continue
        else:
            snps = snp_source

        het_mask = (snps['VAF'] >= het_lo) & (snps['VAF'] <= het_hi)
        hets = snps[het_mask]

        chroms = hets['chromosome'].astype(str).values
        positions = hets['position'].astype(int).values
        vafs = hets['VAF'].astype(float).values

        for chrom, pos, vaf in zip(chroms, positions, vafs):
            pos_vafs[(chrom, pos)].append(vaf)

    baselines = {}
    for key, vafs in pos_vafs.items():
        if len(vafs) >= 2:
            baselines[key] = float(np.median(vafs))
        else:
            baselines[key] = 0.5

    baseline_vals = list(baselines.values())
    print(f"  Built population baselines for {len(baselines)} probe positions "
          f"from {len(control_snp_files)} controls")
    print(f"  Median baseline VAF: {np.median(baseline_vals):.4f} "
          f"(range: {min(baseline_vals):.4f} - {max(baseline_vals):.4f})")

    return baselines

def build_control_file_map(manifest, simulated_dir):
    """
    Build mapping: control_name -> path to one SNP file per control.

    Parameters
    ----------
    manifest : str (path to CSV) or DataFrame
    simulated_dir : str
    """
    if isinstance(manifest, str):
        manifest = pd.read_csv(manifest)

    control_files = {}
    for control in manifest['Sample'].unique():
        ctrl_rows = manifest[manifest['Sample'] == control]
        if len(ctrl_rows) == 0:
            continue
        lrr_file = ctrl_rows.iloc[0]['File_Name']
        snp_file = lrr_file.replace(
            '_PON_normalised_read_depths_and_LRR.txt', '_SNPs.txt')
        snp_path = os.path.join(simulated_dir, snp_file)
        if os.path.exists(snp_path):
            control_files[control] = snp_path
    return control_files

def derive_phasing_from_vaf(snp_data, chrom, start, end, event_type=None,
                            min_hets=3, min_median_ai=0.01,
                            vaf_baselines=None):
    """
    Derive haplotype phasing from a high-CF sample's VAF.

    At high CF, het SNPs are pushed clearly above or below their baseline:
      VAF > baseline -> ALT on amplified haplotype -> phase = 1
      VAF < baseline -> ALT on other haplotype -> phase = 0

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    event_type : str or None - 'CN-LOH', 'GAIN', 'LOSS'
    min_hets : int - minimum het SNPs required
    min_median_ai : float - minimum median AI to accept phasing
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.

    Returns
    -------
    phase_lookup : dict {position: phase} or None if phasing fails
    n_phased : int - number of phased SNPs
    median_ai : float - median allelic imbalance in reference
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]
    region_snps = chrom_snps[
        (chrom_snps['position'] >= start) & (chrom_snps['position'] <= end)
    ]

    # VAF filter depends on event type
    # For CN-LOH/LOSS at high CF, original hets are pushed to VAF ~0 or ~1
    # (they look homozygous). We still need to phase them.
    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        ref_hets = region_snps.copy()
    else:
        ref_hets = region_snps[
            (region_snps['VAF'] >= 0.01) & (region_snps['VAF'] <= 0.99)
        ].copy()

    # deduplicate positions keeping highest AI row
    ref_hets = ref_hets.copy()
    ref_hets['_ai'] = np.abs(ref_hets['VAF'].astype(float) - 0.5)
    ref_hets = (ref_hets.sort_values('_ai', ascending=False)
                        .drop_duplicates(subset='position', keep='first')
                        .drop(columns='_ai'))

    if len(ref_hets) < min_hets:
        return None, len(ref_hets), 0

    # Get baseline for each position
    positions = ref_hets['position'].astype(int).values
    vafs = ref_hets['VAF'].astype(float).values

    if vaf_baselines is not None:
        bl = np.array([vaf_baselines.get((chrom_str, int(pos)), 0.5) for pos in positions])
    else:
        bl = np.full(len(positions), 0.5)

    # Check signal strength (AI relative to baseline)
    ai_values = np.abs(vafs - bl)
    median_ai = float(np.median(ai_values))
    if event_type not in ('CN-LOH', 'CNLOH', 'LOSS') and median_ai < min_median_ai:
        return None, len(ref_hets), median_ai

    # Derive phase relative to baseline
    phases = (vafs > bl).astype(float)
    phase_lookup = dict(zip(positions.tolist(), phases.tolist()))

    return phase_lookup, len(ref_hets), median_ai

def estimate_cf_phased(snp_data, chrom, start, end, phase_lookup, event_type,
                       het_lo=0.2, het_hi=0.8, min_hets=3, p_threshold=0.05,
                       vaf_baselines=None, n_permutations=10000, perm_seed=42,
                       baseline_weight=1.0):
    """
    Estimate cell fraction using externally-derived phasing.

    Computes phased_dev = (2*phase - 1) * (VAF - baseline) for each het SNP,
    then tests whether mean phased_dev is significantly > 0 using both a
    one-sided t-test and a sign-flip permutation test (both must yield
    P < p_threshold for the event to be called significant).

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    phase_lookup : dict {position: phase} from reference sample
    event_type : str ('CN-LOH', 'GAIN', 'LOSS')
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    p_threshold : float - significance threshold (applied to both tests)
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.
    n_permutations : int - number of sign-flip permutations
    perm_seed : int - random seed for permutation test

    Returns
    -------
    dict with cf_estimate, p_onesided, p_permutation, significant, n_hets,
         mean_phased_dev, t_stat
    or None if insufficient data
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]

    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        
        def compute_devs(lo, hi):
            p_hets = chrom_snps[
                (chrom_snps['VAF'] >= lo) &
                (chrom_snps['VAF'] <= hi)
            ].copy()
            p_hets = p_hets[
                (p_hets['position'] >= start) &
                (p_hets['position'] <= end)
            ]
            devs = []
            for pos, vaf in zip(p_hets['position'].astype(int).values,
                                p_hets['VAF'].astype(float).values):
                if pos in phase_lookup:
                    phase = phase_lookup[pos]
                    bl = vaf_baselines.get((chrom_str, int(pos)), 0.5) if vaf_baselines else 0.5
                    bl_partial = 0.5 + baseline_weight * (bl - 0.5)
                    devs.append((2 * phase - 1) * (vaf - bl_partial))
            return devs
        
        region_size_mb = (end - start) / 1e6
        density_threshold = 0.5  # SNPs/Mb

        # Pass 1: 0.2-0.8
        devs = compute_devs(0.2, 0.8)
        pass_used = 1
        eff_het_lo_used, eff_het_hi_used = 0.2, 0.8
        density = len(devs) / region_size_mb

        # Pass 2: 0.05-0.95
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.05, 0.95)
            pass_used = 2
            eff_het_lo_used, eff_het_hi_used = 0.05, 0.95
            density = len(devs) / region_size_mb

        # Pass 3: 0.0-1.0
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.001, 0.999)
            pass_used = 3
            eff_het_lo_used, eff_het_hi_used = 0.001, 0.999

        # Set effective bounds to match whichever pass was used
        # (devs already computed — pass directly to phased_dev array)
        phased_dev = np.array(devs)
        n_hets = len(phased_dev)

        if n_hets < min_hets:
            return None

        # Skip the main het filter block below — already computed
        mean_dev = np.mean(phased_dev)
        t_stat, p_two = ttest_1samp(phased_dev, 0)
        p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2
        rng = np.random.default_rng(perm_seed)
        abs_vals = np.abs(phased_dev)
        null_means = np.array([
            np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
            for _ in range(n_permutations)
        ])
        p_permutation = float(np.mean(null_means >= mean_dev))
        significant = (p_onesided < p_threshold)
        cf_estimate = phased_dev_to_cf(mean_dev, event_type)

        se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
        t_crit = t_dist.ppf(0.975, df=n_hets - 1)
        cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
        cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)

        return {
            'cf_estimate': cf_estimate,
            'cf_lower_95': cf_lower_95,
            'cf_upper_95': cf_upper_95,
            'p_twosided': p_two,
            'p_onesided': p_onesided,
            'p_permutation': p_permutation,
            'significant': significant,
            'n_hets': n_hets,
            'mean_phased_dev': mean_dev,
            't_stat': t_stat,
            'pass_used': pass_used,
            'eff_het_lo': eff_het_lo_used,
            'eff_het_hi': eff_het_hi_used,
        }

    else:
        # GAIN — standard het filter, no multi-pass needed
        pass_used = 1
        eff_het_lo = het_lo
        eff_het_hi = het_hi
        eff_het_lo_used = het_lo
        eff_het_hi_used = het_hi

    hets = chrom_snps[
        (chrom_snps['VAF'] >= eff_het_lo) & (chrom_snps['VAF'] <= eff_het_hi)
    ].copy()

    if len(hets) == 0:
        return None

    # Filter to region + phased positions
    region_hets = hets[
        (hets['position'] >= start) & (hets['position'] <= end)
    ].copy()

    positions = region_hets['position'].astype(int).values
    vafs = region_hets['VAF'].astype(float).values

    phased_dev = []
    for pos, vaf in zip(positions, vafs):
        if pos in phase_lookup:
            phase = phase_lookup[pos]
            if vaf_baselines is not None:
                bl = vaf_baselines.get((chrom_str, int(pos)), 0.5)
            else:
                bl = 0.5
            bl_partial = 0.5 + baseline_weight * (bl-0.5)
            dev = (2 * phase - 1) * (vaf - bl_partial)
            phased_dev.append(dev)

    phased_dev = np.array(phased_dev)
    n_hets = len(phased_dev)

    if n_hets < min_hets:
        return None

    # --- One-sample t-test: is mean phased_dev > 0? ---
    mean_dev = np.mean(phased_dev)
    t_stat, p_two = ttest_1samp(phased_dev, 0)

    # One-sided p-value (we expect positive deviation)
    p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    # --- Sign-flip permutation test ---
    rng = np.random.default_rng(perm_seed)
    abs_vals = np.abs(phased_dev)
    null_means = np.array([
        np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
        for _ in range(n_permutations)
    ])
    p_permutation = float(np.mean(null_means >= mean_dev))

    # --- Significance requires just t-test (as testing in known regions) ---
    # significant = (p_onesided < p_threshold) and (p_permutation < p_threshold)
    significant = (p_onesided < p_threshold)

    # CF estimation
    cf_estimate = phased_dev_to_cf(mean_dev, event_type)
    se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
    t_crit = t_dist.ppf(0.975, df=n_hets - 1)
    cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
    cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)
    return {
        'cf_estimate': cf_estimate,
        'cf_lower_95': cf_lower_95,
        'cf_upper_95': cf_upper_95,
        'p_twosided': p_two,
        'p_onesided': p_onesided,
        'p_permutation': p_permutation,
        'significant': significant,
        'n_hets': n_hets,
        'mean_phased_dev': mean_dev,
        't_stat': t_stat,
        'pass_used': pass_used,      # 1, 2, or 3
        'eff_het_lo': eff_het_lo_used,
        'eff_het_hi': eff_het_hi_used,
    }

In [ ]:
def build_pon_snp_file_map(pon_base_path, pon_libraries, allowed_samples=None):
    """
    Scan PON library directories for control SNP files.
    Returns dict {sample_name: snp_filepath}
    
    Expects structure: base_path/LIBRARY/SAMPLE_NAME/*_SNPs.txt
    """
    import glob
    pon_snp_files = {}
    
    for lib in pon_libraries:
        lib_path = os.path.join(pon_base_path, lib)
        if not os.path.exists(lib_path):
            print(f"  Warning: library path not found: {lib_path}")
            continue
        
        for sample_dir in os.listdir(lib_path):
            if not sample_dir.startswith('CNTRL'):
                continue
            if allowed_samples is not None and sample_dir not in allowed_samples:
                continue
            sample_path = os.path.join(lib_path, sample_dir)
            if not os.path.isdir(sample_path):
                continue
            
            # Search for SNP file
            snp_files = glob.glob(os.path.join(sample_path, '*only_SNPs_annovar_annotated.txt'))
            if not snp_files:
                # try subdirectories
                snp_files = glob.glob(os.path.join(sample_path, '**', '*only_SNPs_annovar_annotated.txt'), recursive=True)
            
            if snp_files:
                pon_snp_files[sample_dir] = snp_files[0]
            else:
                print(f"  Warning: no SNP file found for {sample_dir}")
    
    print(f"  Found SNP files for {len(pon_snp_files)} PON samples")
    return pon_snp_files

def build_pon_baselines(pon_snp_files, known_germline_chroms=None,
                        het_lo=0.01, het_hi=0.99, min_depth = 20,
                        exclude_sample=None):
    """
    Build population VAF baselines from PON SNP files with germline mCA masking.
    
    Parameters
    ----------
    pon_snp_files : dict {sample_name: filepath}
    known_germline_chroms : dict {sample_name: [chroms_to_exclude]}
        Per-sample chromosomes to exclude from baseline contribution
    het_lo, het_hi : VAF range for het filter
    exclude_sample : str or None
        Sample to exclude (for leave-one-out FPR analysis)
    
    Returns
    -------
    baselines : dict {(chrom, position): median_vaf}
    """
    from collections import defaultdict
    
    if known_germline_chroms is None:
        known_germline_chroms = {}
    
    pos_vafs = defaultdict(list)
    n_loaded = 0
    
    for sample_name, snp_path in pon_snp_files.items():
        # Leave-one-out exclusion
        if exclude_sample is not None and sample_name == exclude_sample:
            continue
        
        try:
            snps = pd.read_csv(snp_path, sep='\t')
        except Exception as e:
            print(f"  Warning: could not load {sample_name}: {e}")
            continue
        
        # Get chromosomes to exclude for this sample
        excluded_chroms = set(known_germline_chroms.get(sample_name, []))
        
        # Depth + Het filter
        depth_mask = snps['total_depth'] >=min_depth
        het_mask = (snps['VAF'] >= het_lo) & (snps['VAF'] <= het_hi)
        hets = snps[depth_mask & het_mask].copy()
        
        # Mask germline mCA chromosomes for this sample
        if excluded_chroms:
            hets = hets[~hets['chromosome'].astype(str).isin(excluded_chroms)]
        
        chroms = hets['chromosome'].astype(str).values
        positions = hets['position'].astype(int).values
        vafs = hets['VAF'].astype(float).values
        
        for chrom, pos, vaf in zip(chroms, positions, vafs):
            pos_vafs[(chrom, pos)].append(vaf)
        
        n_loaded += 1
    
    print(f"  Loaded {n_loaded} PON samples"
          + (f" (excluded: {exclude_sample})" if exclude_sample else ""))
    
    # Build median baselines — require at least 3 samples per position
    baselines = {}
    n_insufficient = 0
    for key, vafs in pos_vafs.items():
        if len(vafs) >= 3:
            baselines[key] = float(np.median(vafs))
        else:
            baselines[key] = 0.5
            n_insufficient += 1
    
    baseline_vals = list(baselines.values())
    print(f"  Built baselines for {len(baselines)} positions "
          f"({n_insufficient} positions with <3 samples → defaulted to 0.5)")
    if baseline_vals:
        print(f"  Median baseline VAF: {np.median(baseline_vals):.4f} "
              f"(range: {np.min(baseline_vals):.4f} - {np.max(baseline_vals):.4f})")
    
    return baselines

### Longitudinal phasing pipeline

In [ ]:
def get_sample_filename(manifest, control, event_type, fraction, chrom, start, end):
    """Find the SNP filename for a specific sample in the manifest."""
    type_map = {'CN-LOH': 'CNLOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    sim_type = type_map.get(event_type, event_type)
    
    matches = manifest[
        (manifest['Sample'] == control) &
        (manifest['Type'] == sim_type) &
        (np.isclose(manifest['Fraction_Percent'], fraction)) &
        (manifest['Chromosome'] == chrom) &
        (manifest['Start_bp'] == start) &
        (manifest['End_bp'] == end)
    ]
    
    if len(matches) == 0:
        return None
    
    # Derive SNP filename from the LRR filename
    lrr_file = matches.iloc[0]['File_Name']
    snp_file = lrr_file.replace('_PON_normalised_read_depths_and_LRR.txt', '_SNPs.txt')
    return snp_file

In [ ]:
def run_longitudinal_pipeline(truth_csv, manifest_csv, simulated_dir,
                              reference_cfs=None, target_cfs=None,
                              het_lo=0.02, het_hi=0.98, min_hets=3,
                              min_median_ai=0.01, p_threshold=0.05,
                              vaf_baselines=None,
                              baseline_weight=1.0,
                              debug=True):
    """
    Run the full longitudinal phasing analysis.
    
    For each family × reference_cf:
      - Check if mCA was TP at reference_cf (from unphased caller)
      - If yes, derive phasing from reference sample's VAF
      - Apply phasing to all lower-CF samples
      - Record detection results
    
    Parameters
    ----------
    truth_csv : str - path to truth_comparison.csv
    manifest_csv : str - path to simulation_manifest.csv
    simulated_dir : str - path to simulated sample files
    reference_cfs : list - CF levels to try as phasing reference
    target_cfs : list - CF levels to test detection at
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    min_median_ai : float - minimum median AI to accept phasing
    p_threshold : float - significance threshold
    debug : bool - print progress
    """
    if reference_cfs is None:
        reference_cfs = [1.0, 0.75, 0.5, 0.25]
    if target_cfs is None:
        target_cfs = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]
    
    # Load data
    tc = pd.read_csv(truth_csv)
    manifest = pd.read_csv(manifest_csv)
    
    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map)
    
    # Assign size categories
    tc['size'] = pd.cut(
        tc['Truth_Length_Mb'],
        bins=[0, 2.5, 3.5, 5.5, 10.5, 20.5, 200],
        labels=['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']
    )
    
    # Group into families
    family_cols = ['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End']
    families = tc.groupby(family_cols)
    
    n_families = families.ngroups
    print(f"Total families: {n_families}")
    
    all_results = []
    
    # ══════════════════════════════════════════════════════════════════════
    # DEBUG: Track skip reasons by event type and reference CF
    # ══════════════════════════════════════════════════════════════════════
    from collections import defaultdict
    skip_counts = defaultdict(lambda: defaultdict(int))
    # Keys: (event, ref_cf) -> reason -> count
    proceed_counts = defaultdict(lambda: defaultdict(int))
    
    for fam_idx, (fam_key, fam_df) in enumerate(families):
        control, truth_type, chrom, true_start, true_end = fam_key
        event = event_map[truth_type]
        
        # Get metadata from first row
        first = fam_df.iloc[0]
        size = first['size']
        geometry = first['Truth_Geometry']
        
        if fam_idx % 100 == 0 and debug:
            print(f"  Processing family {fam_idx+1}/{n_families}...")
        
        # ── Try each reference CF independently ──
        for ref_cf in reference_cfs:
            
            debug_key = (event, ref_cf)
            
            # Was this mCA detected at this reference CF?
            ref_row = fam_df[np.isclose(fam_df['Truth_Fraction'], ref_cf)]
            if len(ref_row) == 0:
                skip_counts[debug_key]['no_ref_row_at_cf'] += 1
                continue
            
            ref_row = ref_row.iloc[0]
            
            if ref_row['Status'] != 'TRUE_POSITIVE':
                skip_counts[debug_key]['not_true_positive'] += 1
                continue
            
            # Get DETECTED coordinates (not ground truth)
            det_start = int(ref_row['Detected_Start'])
            det_end = int(ref_row['Detected_End'])
            det_type = ref_row['Detected_Type']
            
            # Find and load the reference SNP file
            ref_snp_name = get_sample_filename(
                manifest, control, event, ref_cf, chrom, true_start, true_end
            )
            if ref_snp_name is None:
                skip_counts[debug_key]['get_sample_filename_returned_None'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: get_sample_filename returned None "
                          f"for {control} {chrom}:{true_start}-{true_end}")
                continue
            
            ref_snp_path = os.path.join(simulated_dir, ref_snp_name)
            if not os.path.exists(ref_snp_path):
                skip_counts[debug_key]['ref_file_not_found'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: ref SNP file not found: {ref_snp_path}")
                continue
            
            try:
                ref_snps = pd.read_csv(ref_snp_path, sep='\t')
            except Exception as e:
                skip_counts[debug_key]['ref_file_read_error'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: error reading ref file: {e}")
                continue
            
            # Derive phasing from reference VAF at DETECTED coordinates
            phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
                ref_snps, chrom, det_start, det_end, event_type=event,
                min_hets=min_hets, min_median_ai=min_median_ai,
                vaf_baselines=vaf_baselines
            )
            
            if phase_lookup is None:
                skip_counts[debug_key]['phasing_failed'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: derive_phasing_from_vaf returned None "
                          f"for {control} {chrom}:{det_start}-{det_end} "
                          f"(n_phased={n_phased}, median_ai={median_ai:.4f})")
                continue
            
            # If we get here, phasing succeeded — count it
            proceed_counts[debug_key]['phasing_succeeded'] += 1
            
            # ── Apply phasing to all target CFs ──
            for target_cf in target_cfs:
                if target_cf >= ref_cf:
                    continue  # Only test lower CFs
                
                target_debug_key = (event, ref_cf, target_cf)
                
                # Find and load target SNP file
                target_snp_name = get_sample_filename(
                    manifest, control, event, target_cf, chrom, true_start, true_end
                )
                if target_snp_name is None:
                    skip_counts[debug_key]['target_filename_None'] += 1
                    continue
                
                target_snp_path = os.path.join(simulated_dir, target_snp_name)
                if not os.path.exists(target_snp_path):
                    skip_counts[debug_key]['target_file_not_found'] += 1
                    continue
                
                try:
                    target_snps = pd.read_csv(target_snp_path, sep='\t')
                except Exception:
                    skip_counts[debug_key]['target_file_read_error'] += 1
                    continue
                
                # Estimate CF using reference-derived phasing at DETECTED coordinates
                result = estimate_cf_phased(
                    target_snps, chrom, det_start, det_end, phase_lookup, event,
                    het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
                    p_threshold=p_threshold,
                    vaf_baselines=vaf_baselines, baseline_weight=baseline_weight
                )
                
                if result is None:
                    all_results.append({
                        'control': control,
                        'event': event,
                        'chromosome': chrom,
                        'true_start': true_start,
                        'true_end': true_end,
                        'det_start': det_start,
                        'det_end': det_end,
                        'size': size,
                        'geometry': geometry,
                        'ref_cf': ref_cf,
                        'target_cf': target_cf,
                        'ref_n_phased': n_phased,
                        'ref_median_ai': median_ai,
                        'significant': False,
                        'cf_estimate': np.nan,
                        'p_onesided': np.nan,
                        'p_twosided': np.nan,
                        'n_hets': 0,
                        'mean_phased_dev': np.nan,
                        't_stat': np.nan,
                        'status': 'insufficient_hets',
                    })
                    continue
                
                all_results.append({
                    'control': control,
                    'event': event,
                    'chromosome': chrom,
                    'true_start': true_start,
                    'true_end': true_end,
                    'det_start': det_start,
                    'det_end': det_end,
                    'size': size,
                    'geometry': geometry,
                    'ref_cf': ref_cf,
                    'target_cf': target_cf,
                    'ref_n_phased': n_phased,
                    'ref_median_ai': median_ai,
                    'significant': result['significant'],
                    'cf_estimate': result['cf_estimate'],
                    'p_onesided': result['p_onesided'],
                    'p_twosided': result['p_twosided'],
                    'n_hets': result['n_hets'],
                    'mean_phased_dev': result['mean_phased_dev'],
                    't_stat': result['t_stat'],
                    'status': 'tested',
                })
    
    # ══════════════════════════════════════════════════════════════════════
    # DEBUG: Print skip reason summary
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "="*80)
    print("DEBUG: FAMILY SKIP REASONS BY EVENT TYPE AND REFERENCE CF")
    print("="*80)
    
    for event in ['CN-LOH', 'GAIN', 'LOSS']:
        print(f"\n  {event}:")
        for ref_cf in (reference_cfs or [1.0, 0.75, 0.5, 0.25]):
            key = (event, ref_cf)
            if key in skip_counts or key in proceed_counts:
                print(f"    ref={ref_cf*100:.0f}% CF:")
                for reason, count in sorted(skip_counts[key].items()):
                    print(f"      SKIPPED - {reason}: {count}")
                for reason, count in sorted(proceed_counts[key].items()):
                    print(f"      OK      - {reason}: {count}")
    
    print("\n" + "="*80)
    print("DEBUG: SUMMARY — families reaching phasing step vs dropped before")
    print("="*80)
    for event in ['CN-LOH', 'GAIN', 'LOSS']:
        for ref_cf in (reference_cfs or [1.0, 0.75, 0.5, 0.25]):
            key = (event, ref_cf)
            succeeded = proceed_counts[key].get('phasing_succeeded', 0)
            total_skipped = sum(skip_counts[key].values())
            # Subtract target-level skips (those happen after phasing succeeded)
            ref_level_skips = sum(v for k, v in skip_counts[key].items() 
                                  if not k.startswith('target_'))
            print(f"  {event:8s} ref={ref_cf*100:>3.0f}%: "
                  f"{succeeded} phased OK, "
                  f"{ref_level_skips} dropped before phasing "
                  f"(breakdown: {dict([(k,v) for k,v in skip_counts[key].items() if not k.startswith('target_')])})")
    
    results_df = pd.DataFrame(all_results)
    return results_df

def summarise_results(results_df, reference_cfs, truth_csv):
    """
    Print summary tables for longitudinal results.
    
    Parameters
    ----------
    results_df : DataFrame - output from run_longitudinal_pipeline
    reference_cfs : list - reference CF levels used
    truth_csv : str - path to truth_comparison.csv (for reference detection rates)
    """
    tested = results_df[results_df['status'] == 'tested']
    
    # ── Summary 1: Detection rate by reference CF and target CF ──
    print("\n" + "="*80)
    print("LONGITUDINAL SENSITIVITY: By reference CF and target CF")
    print("="*80)
    print("(Conditioned on mCA being detected at the reference CF)")
    
    for ref_cf in reference_cfs:
        ref_data = tested[np.isclose(tested['ref_cf'], ref_cf)]
        if len(ref_data) == 0:
            continue
        
        print(f"\n  Reference CF = {ref_cf*100:.0f}%:")
        print(f"  {'Target CF':>12} {'N':>6} {'Detected':>10} {'Sensitivity':>12} {'Mean CF est':>12}")
        print(f"  {'-'*12} {'-'*6} {'-'*10} {'-'*12} {'-'*12}")
        
        for target_cf in sorted(ref_data['target_cf'].unique()):
            subset = ref_data[np.isclose(ref_data['target_cf'], target_cf)]
            n = len(subset)
            n_sig = subset['significant'].sum()
            mean_est = subset.loc[subset['significant'], 'cf_estimate'].mean()
            print(f"  {target_cf*100:>11.1f}% {n:>6} {n_sig:>6}/{n:<3} {n_sig/n*100:>10.1f}% {mean_est:>11.4f}" 
                  if n_sig > 0 else f"  {target_cf*100:>11.1f}% {n:>6} {n_sig:>6}/{n:<3} {n_sig/n*100:>10.1f}% {'N/A':>11}")
    
    # ── Summary 2: By event type ──
    print("\n" + "="*80)
    print("LONGITUDINAL SENSITIVITY: By event type (ref=100% CF)")
    print("="*80)
    
    ref100 = tested[np.isclose(tested['ref_cf'], 1.0)]
    for event in ['CN-LOH', 'GAIN', 'LOSS']:
        evt_data = ref100[ref100['event'] == event]
        if len(evt_data) == 0:
            continue
        print(f"\n  {event}:")
        for target_cf in sorted(evt_data['target_cf'].unique()):
            subset = evt_data[np.isclose(evt_data['target_cf'], target_cf)]
            n = len(subset)
            n_sig = subset['significant'].sum()
            print(f"    Target {target_cf*100:>5.1f}%: {n_sig}/{n} ({n_sig/n*100:.1f}%)")
    
    # ── Summary 3: By size (ref=100% CF) ──
    print("\n" + "="*80)
    print("LONGITUDINAL SENSITIVITY: By event size (ref=100% CF)")
    print("="*80)
    
    for size in ['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']:
        size_data = ref100[ref100['size'] == size]
        if len(size_data) == 0:
            continue
        print(f"\n  {size}:")
        for target_cf in sorted(size_data['target_cf'].unique()):
            subset = size_data[np.isclose(size_data['target_cf'], target_cf)]
            n = len(subset)
            n_sig = subset['significant'].sum()
            print(f"    Target {target_cf*100:>5.1f}%: {n_sig}/{n} ({n_sig/n*100:.1f}%)")
    
    # ── Summary 4: Reference detection rates ──
    tc = pd.read_csv(truth_csv)
    event_map_tc = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map_tc)
    families = tc.groupby(['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End'])
    
    print("\n" + "="*80)
    print("REFERENCE DETECTION RATES (unphased caller)")
    print("="*80)
    
    for ref_cf in reference_cfs:
        n_tp = 0
        for name, grp in families:
            row = grp[np.isclose(grp['Truth_Fraction'], ref_cf)]
            if len(row) > 0 and row.iloc[0]['Status'] == 'TRUE_POSITIVE':
                n_tp += 1
        print(f"  {ref_cf*100:>5.0f}% CF: {n_tp}/{families.ngroups} ({n_tp/families.ngroups*100:.1f}%) families detected")


In [ ]:
print("Running longitudinal phasing pipeline...")
print(f"  Reference CFs: {[f'{cf*100:.0f}%' for cf in REFERENCE_CFS]}")
print(f"  P-value threshold: {P_THRESHOLD}")

# Build population baselines for probe bias correction
print("Building population VAF baselines...")
print("Building PON VAF baselines...")
pon_matrix = pd.read_csv(
    'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv',
    sep='\t', nrows=0  # only load headers, not data
)
FINAL_TIMEPOINT_CONTROLS = [c for c in pon_matrix.columns 
                             if c.startswith('CNTRL')]
print(f"PON samples: {len(FINAL_TIMEPOINT_CONTROLS)}")
pon_snp_files = build_pon_snp_file_map(PON_BASE_PATH, PON_LIBRARIES, allowed_samples=FINAL_TIMEPOINT_CONTROLS)
print(f"Samples found: {sorted(pon_snp_files.keys())}")
print(f"Total: {len(pon_snp_files)}")
vaf_baselines = build_pon_baselines(pon_snp_files, known_germline_chroms=KNOWN_GERMLINE_CHROMS)

# control_snp_files = build_control_file_map(MANIFEST_CSV, SIMULATED_DIR)  # reuse from FPR cell
# vaf_baselines = build_population_baselines(control_snp_files)

longitudinal_results = run_longitudinal_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    reference_cfs=REFERENCE_CFS,
    target_cfs=TARGET_CFS,
    het_lo=TARGET_HET_LO,
    het_hi=TARGET_HET_HI,
    min_hets=MIN_HETS,
    min_median_ai=MIN_MEDIAN_AI,
    p_threshold=P_THRESHOLD,
    vaf_baselines=vaf_baselines,
    baseline_weight=1,
    debug=DEBUG
)

# Save results
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'longitudinal_phased_results.csv')
longitudinal_results.to_csv(output_path, index=False)
print(f"\n✅ Saved {len(longitudinal_results)} results to {output_path}")

# Print summaries
summarise_results(longitudinal_results, reference_cfs=REFERENCE_CFS, truth_csv=TRUTH_CSV)


In [ ]:
# TEST: Run longitudinal pipeline WITHOUT population baseline correction
print("Running WITHOUT population baselines...")

longitudinal_results_no_baseline = run_longitudinal_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    reference_cfs=REFERENCE_CFS,
    target_cfs=TARGET_CFS,
    het_lo=TARGET_HET_LO,
    het_hi=TARGET_HET_HI,
    min_hets=MIN_HETS,
    min_median_ai=MIN_MEDIAN_AI,
    p_threshold=P_THRESHOLD,
    vaf_baselines=None,   # <-- key change
    baseline_weight=0,
    debug=True           # suppress family-level debug to keep output clean
)

summarise_results(longitudinal_results_no_baseline, reference_cfs=REFERENCE_CFS, truth_csv=TRUTH_CSV)

## Assessing false positive rate

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FALSE POSITIVE RATE (FPR) ANALYSIS — Cross-Control Design
# ═══════════════════════════════════════════════════════════════════════════
#
# The spike-in simulation builds each sample from ONE control, modifying
# only the mCA chromosome.  Every other chromosome is the original control
# data unchanged.  So within a family the reference file and target file
# share identical data on null chromosomes → correlated noise → inflated FPR.
#
# Fix: derive phasing from control A's null region, then test on a DIFFERENT
# control B's data on that same region.  Independent sequencing runs →
# independent noise → valid FPR estimate.


# ── Known germline events to exclude ──
KNOWN_EVENT_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

# ── Panel BED file ──
PANEL_BED_FILE = "TWIST_CNV_panel_TE-95031423_h19.bed"
MIN_PROBES_FOR_REGION = 20  # Minimum probes in a null region
NULL_REGIONS_PER_FAMILY = 5  # Number of null regions to test per family


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FALSE POSITIVE RATE (FPR) ANALYSIS — Cross-Control Design
# ═══════════════════════════════════════════════════════════════════════════
#
# Uses derive_phasing_from_vaf, estimate_cf_phased, build_population_baselines,
# and build_control_file_map defined in cell 21 (shared functions).
#
# DO NOT redefine those functions here.
# ═══════════════════════════════════════════════════════════════════════════

def load_panel_bed(bed_file):
    """Load panel BED file and return DataFrame."""
    bed_df = pd.read_csv(bed_file, sep='\t', skiprows=3, header=None,
                         names=['chromosome', 'start', 'end'])
    if not bed_df['chromosome'].iloc[0].startswith('chr'):
        bed_df['chromosome'] = 'chr' + bed_df['chromosome'].astype(str)
    return bed_df

def get_excluded_arms(bed_df, hg19_centromeres, min_probes=50):
    """Identify chromosome arms with insufficient probe coverage."""
    excluded = set()
    for chrom, centro in hg19_centromeres.items():
        chrom_probes = bed_df[bed_df['chromosome'] == chrom]
        p_count = len(chrom_probes[chrom_probes['start'] < centro])
        q_count = len(chrom_probes[chrom_probes['start'] >= centro])
        if p_count < min_probes:
            excluded.add(f"{chrom}p")
        if q_count < min_probes:
            excluded.add(f"{chrom}q")
    return excluded

def generate_null_regions(bed_df, chromosome_sizes, exclude_chroms,
                          mca_size_bp, excluded_arms,
                          n_regions=5, min_probes=20, rng=None):
    """Generate random null regions for FPR testing."""
    if rng is None:
        rng = np.random.default_rng()

    valid_chroms = []
    for c in chromosome_sizes:
        if c in exclude_chroms:
            continue
        if f"{c}p" in excluded_arms and f"{c}q" in excluded_arms:
            continue
        if chromosome_sizes[c] < mca_size_bp + 2_000_000:
            continue
        valid_chroms.append(c)

    if not valid_chroms:
        return []

    null_regions = []
    max_attempts = n_regions * 50
    attempts = 0

    while len(null_regions) < n_regions and attempts < max_attempts:
        attempts += 1
        sizes = np.array([chromosome_sizes[c] for c in valid_chroms], dtype=float)
        probs = sizes / sizes.sum()
        chrom = rng.choice(valid_chroms, p=probs)
        chrom_size = chromosome_sizes[chrom]

        margin = 1_000_000
        max_start = chrom_size - mca_size_bp - margin
        if max_start <= margin:
            continue
        start = int(rng.integers(margin, max_start))
        end = start + mca_size_bp

        region_probes = bed_df[
            (bed_df['chromosome'] == chrom) &
            (bed_df['start'] >= start) &
            (bed_df['end'] <= end)
        ]
        if len(region_probes) < min_probes:
            continue

        overlap = False
        for nc, ns, ne in null_regions:
            if nc == chrom and not (end < ns or start > ne):
                overlap = True
                break
        if overlap:
            continue

        null_regions.append((chrom, start, end))

    return null_regions

def run_longitudinal_fpr_pipeline(truth_csv, manifest_csv, simulated_dir,
                                  bed_file=PANEL_BED_FILE,
                                  reference_cfs=None,
                                  null_regions_per_family=NULL_REGIONS_PER_FAMILY,
                                  het_lo=0.02, het_hi=0.98, min_hets=3,
                                  p_threshold=0.05, seed=42, debug=True,
                                  use_baselines=True,
                                  baseline_weight=1.0,
                                  pon_snp_files=None,
                                  pon_known_germline_chroms=None):
    """
    Cross-control FPR analysis with probe bias correction.

    For each family where phasing succeeded on the real mCA:
      1. Derive phasing from control A's VAF in a null region
      2. Pick a DIFFERENT control B (independent sequencing run)
      3. Test control B's data in the same null region
      4. Independent noise + baseline correction -> valid FPR
    """
    if reference_cfs is None:
        reference_cfs = [1.0, 0.75, 0.5, 0.25]

    rng = np.random.default_rng(seed)

    # ── Load data ──
    tc = pd.read_csv(truth_csv)
    manifest = pd.read_csv(manifest_csv)

    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map)
    tc['size'] = pd.cut(
        tc['Truth_Length_Mb'],
        bins=[0, 2.5, 3.5, 5.5, 10.5, 20.5, 200],
        labels=['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']
    )

    # ── Build control file map and load SNP files ──
    control_files = build_control_file_map(manifest, simulated_dir)
    all_controls = sorted(control_files.keys())
    print(f"Controls with SNP files: {len(all_controls)}")

    print("Pre-loading control SNP files...")
    control_snps = {}
    for ctrl, path in control_files.items():
        try:
            control_snps[ctrl] = pd.read_csv(path, sep='\t')
        except Exception as e:
            print(f"  Warning: could not load {ctrl}: {e}")
    print(f"  Loaded {len(control_snps)} control SNP files")

    # ── Build population baselines ──
    print("Building population VAF baselines...")
    # vaf_baselines = build_population_baselines(control_snps) if use_baselines else None

    # ── Pre-build full PON baselines if not using leave-one-out ──
    # (leave-one-out baselines are built per control_a inside the loop)
    if not use_baselines:
        vaf_baselines_cache = None
        use_loo = False
    elif pon_snp_files is not None:
        print("PON baselines: will use leave-one-out per control...")
        vaf_baselines_cache = None   # built per-family inside loop
        use_loo = True
    else:
        print("Building population VAF baselines from simulated controls...")
        vaf_baselines_cache = build_population_baselines(control_snps)
        use_loo = False

    # ── Load panel BED ──
    print(f"Loading panel BED from {bed_file}...")
    bed_df = load_panel_bed(bed_file)

    hg19_centromeres = {
        'chr1': 125000000, 'chr2': 93300000, 'chr3': 91000000, 'chr4': 50400000,
        'chr5': 48400000, 'chr6': 61000000, 'chr7': 59900000, 'chr8': 45600000,
        'chr9': 49000000, 'chr10': 40200000, 'chr11': 53700000, 'chr12': 35800000,
        'chr13': 17900000, 'chr14': 17600000, 'chr15': 19000000, 'chr16': 36600000,
        'chr17': 24000000, 'chr18': 17200000, 'chr19': 26500000, 'chr20': 27500000,
        'chr21': 13200000, 'chr22': 14700000, 'chrX': 60600000,
    }
    excluded_arms = get_excluded_arms(bed_df, hg19_centromeres, min_probes=50)
    print(f"Excluded arms (< 50 probes): {sorted(excluded_arms)}")

    # ── Group families ──
    family_cols = ['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End']
    families = tc.groupby(family_cols)
    n_families = families.ngroups
    print(f"Total families: {n_families}")
    print(f"Null regions per family: {null_regions_per_family}")
    print(f"Design: cross-control (phasing from control A, testing on control B)")

    all_results = []
    stats = defaultdict(int)
    loo_baseline_cache = {}  

    for fam_idx, (fam_key, fam_df) in enumerate(families):
        control_a, truth_type, chrom, true_start, true_end = fam_key
        event = event_map[truth_type]
        first = fam_df.iloc[0]
        size = first['size']
        geometry = first['Truth_Geometry']
        mca_size_bp = int(true_end - true_start)

        if fam_idx % 100 == 0 and debug:
            print(f"  FPR: family {fam_idx+1}/{n_families} "
                  f"({event}, {size}, {chrom}, {control_a})...")

        if control_a not in control_snps:
            stats['control_a_not_loaded'] += 1
            continue

        other_controls = [c for c in all_controls
                          if c != control_a and c in control_snps]
        if not other_controls:
            stats['no_other_controls'] += 1
            continue

        for ref_cf in reference_cfs:
            ref_row = fam_df[np.isclose(fam_df['Truth_Fraction'], ref_cf)]
            if len(ref_row) == 0:
                continue
            ref_row = ref_row.iloc[0]
            if ref_row['Status'] != 'TRUE_POSITIVE':
                stats['ref_not_tp'] += 1
                continue

            ref_snp_name = get_sample_filename(
                manifest, control_a, event, ref_cf, chrom, true_start, true_end
            )
            if ref_snp_name is None:
                continue
            ref_snp_path = os.path.join(simulated_dir, ref_snp_name)
            if not os.path.exists(ref_snp_path):
                continue
            try:
                ref_snps = pd.read_csv(ref_snp_path, sep='\t')
            except Exception:
                continue

            control_b = rng.choice(other_controls)
            target_snps = control_snps[control_b]

            # Leave-one-out: use cached baseline or build if first time seeing this control_a
            if use_loo:
                if control_a not in loo_baseline_cache:
                    loo_baseline_cache[control_a] = build_pon_baselines(
                        pon_snp_files,
                        known_germline_chroms=pon_known_germline_chroms,
                        exclude_sample=control_a
                    )
                vaf_baselines = loo_baseline_cache[control_a]
            else:
                vaf_baselines = vaf_baselines_cache

            germline_a = set((pon_known_germline_chroms or {}).get(control_a, []))
            germline_b = set((pon_known_germline_chroms or {}).get(control_b, []))
            exclude_chroms = {chrom} | germline_a | germline_b

            null_regions = generate_null_regions(
                bed_df, chromosome_sizes, exclude_chroms, mca_size_bp,
                excluded_arms,
                n_regions=null_regions_per_family, min_probes=MIN_PROBES_FOR_REGION,
                rng=rng
            )

            if len(null_regions) == 0:
                stats['no_null_regions'] += 1
                continue

            stats['families_tested'] += 1

            for null_chrom, null_start, null_end in null_regions:

                # Derive phasing from control A (same function as sensitivity pipeline)
                phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
                    ref_snps, null_chrom, null_start, null_end,
                    event_type=event,
                    min_hets=min_hets, min_median_ai=0,
                    vaf_baselines=vaf_baselines
                )

                if phase_lookup is None:
                    stats['null_phasing_failed'] += 1
                    continue

                stats['null_regions_phased'] += 1

                # Test on control B (same function as sensitivity pipeline)
                result = estimate_cf_phased(
                    target_snps, null_chrom, null_start, null_end,
                    phase_lookup, event,
                    het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
                    p_threshold=p_threshold,
                    vaf_baselines=vaf_baselines, baseline_weight=baseline_weight
                )

                if result is None:
                    all_results.append({
                        'control_a': control_a, 'control_b': control_b,
                        'event': event, 'mca_chromosome': chrom,
                        'null_chromosome': null_chrom,
                        'null_start': null_start, 'null_end': null_end,
                        'size': size, 'geometry': geometry, 'ref_cf': ref_cf,
                        'n_phased_ref': n_phased,
                        'significant': False, 'cf_estimate': np.nan,
                        'p_onesided': np.nan, 'p_twosided': np.nan,
                        'p_permutation': np.nan,
                        'n_hets': 0, 'mean_phased_dev': np.nan,
                        't_stat': np.nan,
                        'status': 'insufficient_hets', 'is_null': True,
                    })
                else:
                    all_results.append({
                        'control_a': control_a, 'control_b': control_b,
                        'event': event, 'mca_chromosome': chrom,
                        'null_chromosome': null_chrom,
                        'null_start': null_start, 'null_end': null_end,
                        'size': size, 'geometry': geometry, 'ref_cf': ref_cf,
                        'n_phased_ref': n_phased,
                        'significant': result['significant'],
                        'cf_estimate': result['cf_estimate'],
                        'p_onesided': result['p_onesided'],
                        'p_twosided': result['p_twosided'],
                        'p_permutation': result['p_permutation'],
                        'n_hets': result['n_hets'],
                        'mean_phased_dev': result['mean_phased_dev'],
                        't_stat': result['t_stat'],
                        'status': 'tested', 'is_null': True,
                    })

                    if result['significant']:
                        stats['false_positives'] += 1
                    stats['null_tests'] += 1

    fpr_results = pd.DataFrame(all_results)

    # ── Summary ──
    print("\n" + "=" * 80)
    print("FPR ANALYSIS SUMMARY (cross-control design)")
    print("=" * 80)
    print(f"  Families tested:        {stats['families_tested']}")
    print(f"  Null regions phased:    {stats['null_regions_phased']}")
    print(f"  Null phasing failed:    {stats['null_phasing_failed']}")
    print(f"  Total null tests:       {stats['null_tests']}")
    print(f"  False positives:        {stats['false_positives']}")
    if stats['null_tests'] > 0:
        overall_fpr = stats['false_positives'] / stats['null_tests'] * 100
        print(f"  Overall FPR:            {overall_fpr:.2f}%")
        print(f"  Expected under null:    ~5% (alpha = 0.05)")

    tested = fpr_results[fpr_results['status'] == 'tested']
    if len(tested) > 0:
        print(f"\n  FPR by reference CF:")
        for rcf in sorted(tested['ref_cf'].unique()):
            rcf_data = tested[np.isclose(tested['ref_cf'], rcf)]
            n = len(rcf_data)
            n_fp = int(rcf_data['significant'].sum())
            print(f"    ref={rcf*100:>5.0f}%: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

        print(f"\n  FPR by event type (all ref CFs):")
        for evt in ['CN-LOH', 'GAIN', 'LOSS']:
            evt_data = tested[tested['event'] == evt]
            if len(evt_data) == 0:
                continue
            n = len(evt_data)
            n_fp = int(evt_data['significant'].sum())
            print(f"    {evt}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

        print(f"\n  FPR by event size (all ref CFs):")
        for sz in ['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']:
            sz_data = tested[tested['size'] == sz]
            if len(sz_data) == 0:
                continue
            n = len(sz_data)
            n_fp = int(sz_data['significant'].sum())
            print(f"    {sz}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

    return fpr_results


In [ ]:
# ═══════════════════════════════════════════════
# RUN FPR ANALYSIS (cross-control design)
# ═══════════════════════════════════════════════
# Insert this cell after the FPR pipeline definition cell

print("Running cross-control FPR analysis...")
fpr_results = run_longitudinal_fpr_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    bed_file=PANEL_BED_FILE,
    reference_cfs=REFERENCE_CFS,
    null_regions_per_family=5,
    het_lo=0.02, het_hi=0.98,
    min_hets=3,
    p_threshold=0.05,
    seed=42,
    debug=True,
    use_baselines=True,
    baseline_weight=BASELINE_WEIGHT,
    pon_snp_files=pon_snp_files,
    pon_known_germline_chroms=KNOWN_GERMLINE_CHROMS,
)

# Save results
fpr_output_path = os.path.join(
    os.path.dirname(OUTPUT_CSV),
    'longitudinal_fpr_results.csv'
)
fpr_results.to_csv(fpr_output_path, index=False)
print(f"\n✅ Saved {len(fpr_results)} FPR results to {fpr_output_path}")


In [ ]:
# ═══════════════════════════════════════════════
# RUN FPR ANALYSIS (cross-control design). WITHOUT BASELINE CORRECTION...
# ═══════════════════════════════════════════════
# Insert this cell after the FPR pipeline definition cell

print("Running cross-control FPR analysis...")
fpr_results = run_longitudinal_fpr_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    bed_file=PANEL_BED_FILE,
    reference_cfs=REFERENCE_CFS,
    null_regions_per_family=5,
    het_lo=0.02, het_hi=0.98,
    min_hets=3,
    p_threshold=0.05,
    seed=42,
    debug=True,
    use_baselines=True,
    baseline_weight=BASELINE_WEIGHT,    # set this in config once sweep done
    pon_snp_files=pon_snp_files,
    pon_known_germline_chroms=KNOWN_GERMLINE_CHROMS,
)

# Save results
fpr_output_path = os.path.join(
    os.path.dirname(OUTPUT_CSV),
    'longitudinal_fpr_results_without_baseline_correction.csv'
)
fpr_results.to_csv(fpr_output_path, index=False)
print(f"\n✅ Saved {len(fpr_results)} FPR results to {fpr_output_path}")


### Assesing differnt baseline correction weights...

In [ ]:
import pandas as pd

# ── Baseline weights to test ──────────────────────────────────────────────
weights_to_test = [0.0, 0.25, 0.5, 0.75, 1.0]

# Key target CFs to report sensitivity at
report_cfs = [0.01, 0.02, 0.05, 0.10, 0.25]

sweep_results = []

pon_snp_files = build_pon_snp_file_map(
    PON_BASE_PATH, PON_LIBRARIES,
    allowed_samples=FINAL_TIMEPOINT_CONTROLS
)
# Build full PON baselines for sensitivity (no leave-one-out needed)
vaf_baselines_pon = build_pon_baselines(
    pon_snp_files,
    known_germline_chroms=KNOWN_GERMLINE_CHROMS
)

for w in weights_to_test:
    print(f"\n{'='*60}")
    print(f"  baseline_weight = {w}")
    print(f"{'='*60}")

    # ── Sensitivity run ───────────────────────────────────────────
    sens_results = run_longitudinal_pipeline(
        truth_csv=TRUTH_CSV,
        manifest_csv=MANIFEST_CSV,
        simulated_dir=SIMULATED_DIR,
        reference_cfs=[1.0],          # ref=100% only to save time
        target_cfs=TARGET_CFS,
        het_lo=TARGET_HET_LO,
        het_hi=TARGET_HET_HI,
        min_hets=MIN_HETS,
        min_median_ai=MIN_MEDIAN_AI,
        p_threshold=P_THRESHOLD,
        vaf_baselines=vaf_baselines_pon,
        baseline_weight=w,
        debug=True
    )

    # ── FPR run ───────────────────────────────────────────────────
    fpr_results = run_longitudinal_fpr_pipeline(
        truth_csv=TRUTH_CSV,
        manifest_csv=MANIFEST_CSV,
        simulated_dir=SIMULATED_DIR,
        bed_file=PANEL_BED_FILE,
        reference_cfs=[1.0],
        null_regions_per_family=5,
        het_lo=0.02, het_hi=0.98,
        min_hets=3,
        p_threshold=0.05,
        seed=42,
        debug=True,
        use_baselines=True,
        baseline_weight=w,
        pon_snp_files=pon_snp_files,
        pon_known_germline_chroms=KNOWN_GERMLINE_CHROMS,
    )

    # # ── Extract sensitivity at key CFs ────────────────────────────
    # row = {'baseline_weight': w}

    # truth = pd.read_csv(TRUTH_CSV)
    # for cf in report_cfs:
    #     subset = sens_results[
    #         (sens_results['ref_cf'] == 1.0) &
    #         (sens_results['target_cf'] == cf)
    #     ]
    #     if len(subset) == 0:
    #         row[f'sens_{int(cf*100)}pct'] = None
    #         continue
    #     detected = subset['significant'].sum()
    #     total = len(subset)
    #     row[f'sens_{int(cf*100)}pct'] = f"{detected}/{total} ({100*detected/total:.1f}%)"

    # ── Extract sensitivity at key CFs ────────────────────────────
    row = {'baseline_weight': w}
    for cf in report_cfs:
        subset = sens_results[
            (sens_results['ref_cf'] == 1.0) &
            (sens_results['target_cf'] == cf)
        ]
        if len(subset) == 0:
            row[f'sens_{int(cf*100)}pct'] = None
            row[f'sens_{int(cf*100)}pct_n'] = None
            continue
        detected = int(subset['significant'].sum())
        total = len(subset)
        row[f'sens_{int(cf*100)}pct'] = f"{detected}/{total} ({100*detected/total:.1f}%)"
        row[f'sens_{int(cf*100)}pct_n'] = 100 * detected / total

    # ── Extract FPR ───────────────────────────────────────────────
    tested = fpr_results[fpr_results['status'] == 'tested'] if 'status' in fpr_results.columns else fpr_results
    total_tests = len(tested)
    fp = int(tested['significant'].sum()) if 'significant' in tested.columns else None
    row['FPR'] = f"{fp}/{total_tests} ({100*fp/total_tests:.1f}%)" if fp is not None else "N/A"
    row['FPR_n'] = 100 * fp / total_tests if fp is not None and total_tests > 0 else None

    sweep_results.append(row)
    print(f"  FPR: {row['FPR']}")
    for cf in report_cfs:
        key = f'sens_{int(cf*100)}pct'
        print(f"  Sensitivity at {int(cf*100)}%: {row[key]}")

# ── Summary table + save ──────────────────────────────────────────────────
print(f"\n\n{'='*60}")
print("SWEEP SUMMARY")
print(f"{'='*60}")
sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

sweep_output = os.path.join(OUTPUT_DIR, 'baseline_weight_sweep.csv')
sweep_df.to_csv(sweep_output, index=False)
print(f"\n✅ Saved sweep results to {sweep_output}")

## Within-family FPR analysis

In [ ]:
def run_within_family_fpr_pipeline(
        truth_csv, manifest_csv, simulated_dir,
        original_snp_files,
        reference_cf=0.5,
        het_lo=0.02, het_hi=0.98,
        min_hets=MIN_HETS,
        min_median_ai=MIN_MEDIAN_AI,
        p_threshold=P_THRESHOLD,
        vaf_baselines=None,
        baseline_weight=1.0,
        known_germline_chroms=None,
        debug=True):
    
    """
    Within-family FPR analysis.

    For each family where the mCA was detected at reference_cf:
      1. Derive phasing from the simulated reference_cf sample (same individual)
      2. Test the original unmodified control SNP file (0% CF) in the same region
      3. Any significant result = false positive

    This mirrors the exact clinical scenario: phasing from a high-CF timepoint,
    testing whether the mCA is detectable in an earlier null timepoint.
    No cross-individual VAF differences, so baseline correction is not needed.
    """

    if known_germline_chroms is None:
        known_germline_chroms = {}

    tc = pd.read_csv(truth_csv)
    manifest = pd.read_csv(manifest_csv)

    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map)
    tc['size'] = pd.cut(
        tc['Truth_Length_Mb'],
        bins=[0, 2.5, 3.5, 5.5, 10.5, 20.5, 200],
        labels=['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']
    )

    # ── Group families ──
    family_cols = ['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End']
    families = tc.groupby(family_cols)
    n_families = families.ngroups
    print(f"Total families: {n_families}")
    print(f"Reference CF for phasing: {reference_cf*100:.0f}%")
    print(f"Null target: original unmodified control (0% CF)")

    all_results = []
    stats = defaultdict(int)

    for fam_idx, (fam_key, fam_df) in enumerate(families):
        control, truth_type, chrom, true_start, true_end = fam_key
        event = event_map[truth_type]
        first = fam_df.iloc[0]
        size = first['size']
        geometry = first['Truth_Geometry']

        if fam_idx % 100 == 0 and debug:
            print(f"  FPR: family {fam_idx+1}/{n_families} "
                  f"({event}, {size}, {chrom}, {control})...")

        # ── Check mCA was detected at reference CF ──
        ref_row = fam_df[np.isclose(fam_df['Truth_Fraction'], reference_cf)]
        if len(ref_row) == 0:
            stats['no_ref_row'] += 1
            continue
        ref_row = ref_row.iloc[0]
        if ref_row['Status'] != 'TRUE_POSITIVE':
            stats['ref_not_tp'] += 1
            continue

        # ── Skip known germline mCA chromosomes ──
        germline_chroms = set(known_germline_chroms.get(control, []))
        if chrom in germline_chroms:
            stats['skipped_germline'] += 1
            continue

        # ── Load simulated reference SNP file ──
        ref_snp_name = get_sample_filename(
            manifest, control, event, reference_cf, chrom, true_start, true_end
        )
        if ref_snp_name is None:
            stats['ref_file_not_found'] += 1
            continue
        ref_snp_path = os.path.join(simulated_dir, ref_snp_name)
        if not os.path.exists(ref_snp_path):
            stats['ref_file_not_found'] += 1
            continue
        try:
            ref_snps = pd.read_csv(ref_snp_path, sep='\t')
        except Exception as e:
            stats['ref_file_not_found'] += 1
            continue

        # ── Find original unmodified control SNP file (0% CF) ──
        if control not in original_snp_files:
            stats['original_not_found'] += 1
            continue
        try:
            original_snps = pd.read_csv(original_snp_files[control], sep='\t')
        except Exception as e:
            stats['original_not_found'] += 1
            continue

        # ── Derive phasing from simulated reference sample ──
        phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
            ref_snps, chrom, true_start, true_end,
            event_type=event,
            min_hets=min_hets, min_median_ai=min_median_ai,
            vaf_baselines=vaf_baselines
        )

        if phase_lookup is None:
            stats['phasing_failed'] += 1
            continue

        stats['families_tested'] += 1

        # ── Test original control in same region ──
        result = estimate_cf_phased(
            original_snps, chrom, true_start, true_end,
            phase_lookup, event,
            het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
            p_threshold=p_threshold,
            vaf_baselines=vaf_baselines,
            baseline_weight=baseline_weight
        )

        if result is None:
            all_results.append({
                'control': control, 'event': event,
                'chromosome': chrom, 'start': true_start, 'end': true_end,
                'size': size, 'geometry': geometry,
                'ref_cf': reference_cf, 'n_phased_ref': n_phased,
                'median_ai_ref': median_ai,
                'significant': False, 'cf_estimate': np.nan,
                'p_onesided': np.nan, 'p_twosided': np.nan,
                'p_permutation': np.nan, 'n_hets': 0,
                'mean_phased_dev': np.nan, 't_stat': np.nan,
                'status': 'insufficient_hets',
            })
        else:
            all_results.append({
                'control': control, 'event': event,
                'chromosome': chrom, 'start': true_start, 'end': true_end,
                'size': size, 'geometry': geometry,
                'ref_cf': reference_cf, 'n_phased_ref': n_phased,
                'median_ai_ref': median_ai,
                'significant': result['significant'],
                'cf_estimate': result['cf_estimate'],
                'p_onesided': result['p_onesided'],
                'p_twosided': result['p_twosided'],
                'p_permutation': result['p_permutation'],
                'n_hets': result['n_hets'],
                'mean_phased_dev': result['mean_phased_dev'],
                't_stat': result['t_stat'],
                'status': 'tested',
            })

            if result['significant']:
                stats['false_positives'] += 1
            stats['null_tests'] += 1

    fpr_results = pd.DataFrame(all_results)

    # ── Summary ──
    print("\n" + "=" * 80)
    print("WITHIN-FAMILY FPR SUMMARY")
    print("=" * 80)
    print(f"  Families tested:        {stats['families_tested']}")
    print(f"  Phasing failed:         {stats['phasing_failed']}")
    print(f"  Original not found:     {stats['original_not_found']}")
    print(f"  Skipped germline:       {stats['skipped_germline']}")
    print(f"  Total null tests:       {stats['null_tests']}")
    print(f"  False positives:        {stats['false_positives']}")
    if stats['null_tests'] > 0:
        overall_fpr = stats['false_positives'] / stats['null_tests'] * 100
        print(f"  Overall FPR:            {overall_fpr:.2f}%")
        print(f"  Expected under null:    ~5% (alpha = 0.05)")

    tested = fpr_results[fpr_results['status'] == 'tested']
    if len(tested) > 0:
        print(f"\n  FPR by event type:")
        for evt in ['CN-LOH', 'GAIN', 'LOSS']:
            evt_data = tested[tested['event'] == evt]
            if len(evt_data) == 0:
                continue
            n = len(evt_data)
            n_fp = int(evt_data['significant'].sum())
            print(f"    {evt}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

        print(f"\n  FPR by event size:")
        for sz in ['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']:
            sz_data = tested[tested['size'] == sz]
            if len(sz_data) == 0:
                continue
            n = len(sz_data)
            n_fp = int(sz_data['significant'].sum())
            print(f"    {sz}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

    return fpr_results

In [ ]:
# Find original (unmodified) control SNP files — used as 0% CF null targets
original_snp_files = build_pon_snp_file_map(
    PON_BASE_PATH, PON_LIBRARIES,
    allowed_samples=FINAL_TIMEPOINT_CONTROLS
)

all_fpr_results = []
for ref_cf in [0.25, 0.5, 0.75, 1.0]:
    results = run_within_family_fpr_pipeline(
        truth_csv=TRUTH_CSV,
        manifest_csv=MANIFEST_CSV,
        simulated_dir=SIMULATED_DIR,
        original_snp_files=original_snp_files,
        reference_cf=ref_cf,
        het_lo=TARGET_HET_LO,
        het_hi=TARGET_HET_HI,
        min_hets=MIN_HETS,
        min_median_ai=MIN_MEDIAN_AI,
        p_threshold=P_THRESHOLD,
        vaf_baselines=None,
        baseline_weight=1.0,
        known_germline_chroms=KNOWN_GERMLINE_CHROMS,
        debug=True
    )
    all_fpr_results.append(results)

# Save results
within_fpr_path = os.path.join(OUTPUT_DIR, 'Within_family_fpr_results.csv')
within_fpr_results = pd.concat(all_fpr_results, ignore_index=True)
within_fpr_results.to_csv(within_fpr_path, index=False)
print(f"\n✅ Saved {len(within_fpr_results)} results to {within_fpr_path}")

In [ ]:
TARGET_CFS

In [ ]:
sens_results = run_longitudinal_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    reference_cfs=[0.25, 0.5, 0.75, 1.0],
    target_cfs=TARGET_CFS,
    het_lo=TARGET_HET_LO,
    het_hi=TARGET_HET_HI,
    min_hets=MIN_HETS,
    min_median_ai=MIN_MEDIAN_AI,
    p_threshold=P_THRESHOLD,
    vaf_baselines=None,
    baseline_weight=1.0,
    debug=True
)

report_cfs = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.20, 0.25, 0.5, 0.75]

for ref_cf in [0.25, 0.5, 0.75, 1.0]:
    print(f"\n{'='*80}")
    print(f"SENSITIVITY SUMMARY (ref={ref_cf*100:.0f}%)")
    print('='*80)
    tested = sens_results[np.isclose(sens_results['ref_cf'], ref_cf)]

    print(f"\n  Overall:")
    for cf in report_cfs:
        subset = tested[np.isclose(tested['target_cf'], cf)]
        if len(subset) == 0:
            continue
        detected = int(subset['significant'].sum())
        total = len(subset)
        print(f"    {cf*100:>5.1f}% CF: {detected}/{total} ({100*detected/total:.1f}%)")

    for evt in ['CN-LOH', 'GAIN', 'LOSS']:
        print(f"\n  {evt}:")
        evt_data = tested[tested['event'] == evt]
        for cf in report_cfs:
            subset = evt_data[np.isclose(evt_data['target_cf'], cf)]
            if len(subset) == 0:
                continue
            detected = int(subset['significant'].sum())
            total = len(subset)
            print(f"    {cf*100:>5.1f}% CF: {detected}/{total} ({100*detected/total:.1f}%)")

# Save
sens_output = os.path.join(OUTPUT_DIR, 'sensitivity_results_all_ref_cfs.csv')
sens_results.to_csv(sens_output, index=False)
print(f"\n✅ Saved to {sens_output}")

---
# Real Samples — Longitudinal Phased mCA Caller
For each mCA listed in `mCAs_for_phasing.csv`:
1. Derive phasing from the reference sample (the timepoint listed in the CSV)
2. Find all other timepoints for that patient across all library folders
3. Apply phased caller to every other timepoint
4. Plot and collate results

In [ ]:
MCAS_CSV = 'CNV_panel_mCA_calling_results/mCAs_for_phasing.csv'

# All library folders to search for timepoints
ALL_LIBRARIES = [
    "SLX_19285",
    "SLX_20125",
    "SLX_20127",
    "SLX_20199",
    "SLX_20201",
    "SLX_20203",
    "SLX_20550",
    "SLX_20566",
    "SLX_20567",
    "SLX_20569",
]

# Output directory for plots
PLOT_DIR = 'CNV_panel_mCA_calling_results/Longitudinal_phased_mCA_calls/PDFs'
os.makedirs(PLOT_DIR, exist_ok=True)

# Phasing parameters
# HET_LO        = 0.02
# HET_HI        = 0.98
HET_LO        = 0.2
HET_HI        = 0.8
MIN_HETS      = 3
MIN_MEDIAN_AI = 0.01
P_THRESHOLD   = 0.05


In [ ]:
# File finding functions

def parse_sample(sample_name):
    """
    Extract (patient_id, timepoint) from a sample name.
    Handles folder suffixes like 'C92_012_s7_CNV_xGenUDI1' -> ('C92_012', 7).
    """
    m = re.match(r'^((?:C92|CNTRL)_\d+)_[sS](\d+)', sample_name.strip())
    if m:
        return m.group(1), int(m.group(2))
    return None, None

def find_sample_folder(library, patient_id, timepoint):
    """
    Find the sample folder for patient_id at timepoint within library.
    Requires folder to start with exactly 'patient_id_s{tp}_' or equal 'patient_id_s{tp}'.
    Prefers non-_CNV_ folder when both exist.
    """
    prefix_lo = f"{patient_id}_s{timepoint}"
    prefix_hi = f"{patient_id}_S{timepoint}"
    candidates = [
        e for e in os.listdir(library)
        if os.path.isdir(os.path.join(library, e))
        and (
            e == prefix_lo or e.startswith(prefix_lo + '_') or
            e == prefix_hi or e.startswith(prefix_hi + '_')
        )
    ]
    if not candidates:
        return None
    non_cnv = [c for c in candidates if 'CNV' not in c]
    return non_cnv[0] if non_cnv else candidates[0]

def find_snp_file(library, folder_name):
    """
    Find the SNP file for a sample. Looks inside the sample subfolder first,
    then falls back to the library root (some samples have SNP files there).
    Returns full path or None.
    """
    SNP_SUFFIX = 'variant_calling_only_SNPs_annovar_annotated.txt'
    sample_dir = os.path.join(library, folder_name)

    # Search inside subfolder
    path = next(
        (os.path.join(sample_dir, f) for f in os.listdir(sample_dir)
         if f.endswith(SNP_SUFFIX)), None
    )
    # Fallback: library root
    if path is None:
        prefix = folder_name.split('_CNV')[0]
        path = next(
            (os.path.join(library, f) for f in os.listdir(library)
             if f.endswith(SNP_SUFFIX) and f.startswith(prefix)), None
        )
    return path

def load_snp_file(path):
    """Load a SNP file, normalising column names for the phased caller (needs 'VAF')."""
    df = pd.read_csv(path, sep='\t')
    # Unphased caller saves as BAF; phased caller needs VAF
    if 'VAF' not in df.columns and 'BAF' in df.columns:
        df = df.rename(columns={'BAF': 'VAF'})
    df['chromosome'] = df['chromosome'].astype(str)
    df['position']   = pd.to_numeric(df['position'], errors='coerce')
    df['VAF']        = pd.to_numeric(df['VAF'],      errors='coerce')
    
    # One row per genomic position — keep highest AI row
    df['_ai'] = np.abs(df['VAF'] - 0.5)
    df = (df.sort_values('_ai', ascending=False)
            .drop_duplicates(subset=['chromosome', 'position'], keep='first')
            .drop(columns='_ai')
            .reset_index(drop=True))
    return df

def find_all_timepoints(patient_id, libraries):
    """
    Search all libraries for every available timepoint for patient_id.
    Returns list of dicts: {timepoint, library, folder, snp_path}
    """
    found = {}  # timepoint -> entry (deduplicate, keep first non-CNV found)
    for lib in libraries:
        if not os.path.isdir(lib):
            continue
        # Scan all folders that match this patient
        for entry in os.listdir(lib):
            full = os.path.join(lib, entry)
            if not os.path.isdir(full):
                continue
            pid, tp = parse_sample(entry)
            if pid != patient_id:
                continue
            snp_path = find_snp_file(lib, entry)
            if snp_path is None:
                continue
            # Prefer non-CNV folder; don't overwrite if already have non-CNV
            if tp not in found or 'CNV' in found[tp]['folder']:
                found[tp] = {
                    'timepoint': tp,
                    'library':   lib,
                    'folder':    entry,
                    'snp_path':  snp_path,
                }
    return sorted(found.values(), key=lambda x: x['timepoint'])


### Run phased longitudinal pipeline on real samples

In [ ]:
mcas = pd.read_csv(MCAS_CSV)

all_results = []

for _, mca_row in mcas.iterrows():
    ref_sample_raw = str(mca_row['sample'])   # e.g. 'C92_022_s7' or 'C92_012_s7_CNV_xGenUDI1'
    chrom          = str(mca_row['chromosome'])
    start          = int(mca_row['start_pos'])
    end            = int(mca_row['end_pos'])
    event          = str(mca_row['event'])

    patient_id, ref_tp = parse_sample(ref_sample_raw)
    if patient_id is None:
        print(f"⚠️  Could not parse sample name: {ref_sample_raw}")
        continue

    print(f"\n{'='*65}")
    print(f"Patient: {patient_id}  |  {chrom} {event} {start/1e6:.1f}–{end/1e6:.1f} Mb  |  ref timepoint: s{ref_tp}")

    # ── 1. Find all timepoints for this patient ──────────────────────────
    timepoints = find_all_timepoints(patient_id, ALL_LIBRARIES)
    if not timepoints:
        print(f"  ⚠️  No timepoint files found for {patient_id}")
        continue

    tps_found = [f"s{t['timepoint']} ({t['library']})" for t in timepoints]
    print(f"  Timepoints found: {tps_found}")

    # ── 2. Load reference sample SNP file ───────────────────────────────
    ref_entry = next((t for t in timepoints if t['timepoint'] == ref_tp), None)
    if ref_entry is None:
        print(f"  ⚠️  Reference timepoint s{ref_tp} not found on disk for {patient_id}")
        continue

    ref_snps = load_snp_file(ref_entry['snp_path'])
    print(f"  Reference: {ref_entry['folder']} ({len(ref_snps)} SNPs)")

    # ── 3. Derive phasing from reference sample ──────────────────────────
    phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
        ref_snps, chrom, start, end, event_type=event,
        min_hets=MIN_HETS, min_median_ai=MIN_MEDIAN_AI,
        vaf_baselines=None  # no population baseline for real samples (yet)
    )

    if phase_lookup is None:
        print(f"  ⚠️  Phasing failed for reference s{ref_tp} "
              f"(n_phased={n_phased}, median_ai={median_ai:.3f})")
        all_results.append({
            'patient': patient_id, 'chromosome': chrom,
            'start_pos': start, 'end_pos': end, 'event': event,
            'ref_timepoint': f"s{ref_tp}", 'ref_library': ref_entry['library'],
            'timepoint': f"s{ref_tp}", 'library': ref_entry['library'],
            'role': 'reference',
            'phase_derived': False, 'n_phased': n_phased, 'median_ai_ref': median_ai,
            'n_hets': None, 'cf_estimate': None, 'p_onesided': None, 'significant': None,
        })
        continue

    print(f"  ✅ Phasing derived: {n_phased} SNPs, median AI={median_ai:.3f}")

    # # ── 4. Determine favored haplotype (majority phase = 1) ──────────────
    # phase_vals = list(phase_lookup.values())
    # favored_haplotype = 1 if sum(phase_vals) >= len(phase_vals) / 2 else 0

    # ── 5. Apply phased caller to ALL timepoints (including reference) ───
    for tp_entry in timepoints:
        tp        = tp_entry['timepoint']
        lib       = tp_entry['library']
        folder    = tp_entry['folder']
        role      = 'reference' if tp == ref_tp else 'earlier' if tp < ref_tp else 'later'

        snps = load_snp_file(tp_entry['snp_path'])

        cf_result = estimate_cf_phased(
            snps, chrom, start, end, phase_lookup, event,
            het_lo=HET_LO, het_hi=HET_HI, min_hets=MIN_HETS,
            p_threshold=P_THRESHOLD, vaf_baselines=None
        )

        if cf_result is None:
            print(f"  s{tp} ({role}): insufficient phased hets")
            cf_est = p_val = t_st = n_h = None
            sig = False
        else:
            cf_est = cf_result['cf_estimate']
            p_val  = cf_result['p_onesided']
            t_st   = cf_result['t_stat']
            n_h    = cf_result['n_hets']
            sig    = cf_result['significant']
            print(f"  s{tp} ({role}): CF={cf_est:.3f}, p={p_val:.4f}, "
                  f"n_hets={n_h}, {'✅ SIG' if sig else '— ns'}")

        all_results.append({
            'patient':       patient_id,
            'chromosome':    chrom,
            'start_pos':     start,
            'end_pos':       end,
            'event':         event,
            'ref_timepoint': f"s{ref_tp}",
            'ref_library':   ref_entry['library'],
            'timepoint':     f"s{tp}",
            'library':       lib,
            'role':          role,
            'phase_derived': True,
            'n_phased':      n_phased,
            'median_ai_ref': median_ai,
            'n_hets':        n_h,
            'cf_estimate':   cf_est,
            'p_onesided':    p_val,
            't_stat':        t_st,
            'significant':   sig,
        })

        # Remap keys from estimate_cf_phased to what plot_known_region_phased expects
        if cf_result is not None:
            plot_cf_result = cf_result.copy()
            p_display = cf_result['p_onesided']
            plot_cf_result['p_onesample']   = p_display
            plot_cf_result['p_value']       = p_display
            plot_cf_result['p_permutation'] = p_display
            plot_cf_result['n_hets_inside']    = cf_result['n_hets']
            plot_cf_result['is_significant']   = cf_result['significant']
            plot_cf_result['significant_raw']  = cf_result['significant']  # ← this was missing
        else:
            plot_cf_result = None

        # ── 6. Plot ─────────────────────────────────────────────────────
        plot_fname = (f"{PLOT_DIR}/{patient_id}_{chrom}_{event}"
                      f"_ref_s{ref_tp}_target_s{tp}.pdf")
        try:
            fig = plot_known_region_smaller_plot(
                snps, chrom, start, end, event,
                phase_lookup,
                cf_result=plot_cf_result,
                sample_name=f"{folder}  ({role})",
                het_lo=HET_LO, het_hi=HET_HI,
                chromosome_sizes=chromosome_sizes,
                ideogram_file=ideogram_file,
                save_path=plot_fname
            )
            if fig is not None:
                plt.close(fig)
        except Exception as e:
            print(f"    ⚠️  Plot failed for s{tp}: {e}")

results_df = pd.DataFrame(all_results)
print(f"\n{'='*65}")
print(f"Done. {len(results_df)} result rows across {results_df['patient'].nunique()} patients.")
display(results_df)


### Summary table and save plots

In [ ]:
# Wide-format: one row per patient/mCA, columns = timepoints
wide_rows = []
for (patient, chrom, event, ref_tp), grp in results_df.groupby(
        ['patient', 'chromosome', 'event', 'ref_timepoint']):

    row = {'patient': patient, 'chromosome': chrom,
           'event': event, 'ref_timepoint': ref_tp,
           'n_phased_snps': grp['n_phased'].iloc[0],
           'median_ai_ref': grp['median_ai_ref'].iloc[0]}

    for _, r in grp.sort_values('timepoint').iterrows():
        tp_label = r['timepoint']
        if r['role'] == 'reference':
            tp_label += ' (REF)'
        if r['cf_estimate'] is not None:
            tag = ' *' if r['significant'] else ''
            row[tp_label] = f"CF={r['cf_estimate']:.2f}{tag}"
        else:
            row[tp_label] = 'no data'

    wide_rows.append(row)

wide_df = pd.DataFrame(wide_rows)

# Sort timepoint columns numerically
def _tp_sort(col):
    m = re.search(r's(\d+)', col)
    return int(m.group(1)) if m else 999
fixed_cols = ['patient', 'chromosome', 'event', 'ref_timepoint',
              'n_phased_snps', 'median_ai_ref']
tp_cols = sorted([c for c in wide_df.columns if c not in fixed_cols], key=_tp_sort)
wide_df = wide_df[fixed_cols + tp_cols]

print("Legend: CF=X.XX = cell fraction estimate  |  * = significant (p<0.05)\n")
display(wide_df)

# Save both formats
results_df.to_csv('CNV_panel_mCA_calling_results/Longitudinal_phased_mCA_calls/mCA_phased_longitudinal_long.csv', index=False)
wide_df.to_csv('CNV_panel_mCA_calling_results/Longitudinal_phased_mCA_calls/mCA_phased_longitudinal_wide.csv', index=False)
print("\nSaved: mCA_phased_longitudinal_long.csv")
print("Saved: mCA_phased_longitudinal_wide.csv")
print(f"Plots: {PLOT_DIR}/")
